# AI Detective: A Knowledge-Based Mystery Solving Agent
CSE518 Artificial Intelligence, Monsoon 2026

The agent solves a fictional murder case. First a CSP removes every possibility that goes against the clues,
then logic and probability are used to find the most likely answer.

In [1]:
import time
from itertools import product

## 1. The Case

Vikram Shah was killed on the night of his 60th birthday party at Riverside Manor. His body was found the next
morning near the staircase, but it had been moved there after the crime. We have to find out who killed him,
with which weapon, where and when.

- Suspects: Anita (business partner), Rohan (nephew), Meera (secretary), Kabir (chef)
- Weapons: Knife, Poison, Trophy, Scarf
- Locations: Library, Kitchen, Garden, Balcony
- Times: 8 PM, 9 PM, 10 PM

**Clues (confirmed facts)**
1. The autopsy found no poison in the body.
2. It rained all evening and there were no footprints in the garden.
3. Vikram gave a speech to all the guests at 8 PM, so he was alive at 8 PM.
4. No knife ever left the kitchen.
5. Kitchen helpers were in the kitchen all evening except at 8 PM, when they went to hear the speech.
6. CCTV shows that Kabir never left the kitchen.
7. The trophy is heavy and never left the library.
8. The library was locked at 9:30 PM.
9. At 9 PM a lot of guests were on the balcony watching fireworks.
10. The scarf was kept in the office and only Anita and Meera had the office key.
11. Anita has a broken wrist and cannot lift the trophy.
12. Meera was on a recorded phone call at 9 PM.
13. Anita left the party at 9:45 PM.

**Uncertain evidence** (not confirmed, used in the Bayesian part)
- Rohan has gambling debts and gets money from Vikram's will.
- A partial fingerprint on the trophy is around a 70% match with Rohan.
- Anita and Vikram had an argument in the morning.
- A guest thinks they saw someone on the balcony around 10 PM, but it was dark.

In [2]:
SUSPECTS  = ["Anita", "Rohan", "Meera", "Kabir"]
WEAPONS   = ["Knife", "Poison", "Trophy", "Scarf"]
LOCATIONS = ["Library", "Kitchen", "Garden", "Balcony"]
TIMES     = ["8 PM", "9 PM", "10 PM"]

ROLES = {"Anita": "business partner", "Rohan": "nephew",
         "Meera": "secretary", "Kabir": "chef"}

TOTAL_HYPOTHESES = len(SUSPECTS) * len(WEAPONS) * len(LOCATIONS) * len(TIMES)
print("Total possible hypotheses:", TOTAL_HYPOTHESES)

Total possible hypotheses: 192


## 2. CSP Formulation

- **Variables:** Culprit, Weapon, Location, Time
- **Domains:** the suspects, weapons, locations and times listed above
- **Constraints:** the 13 clues. Clues 1-3 are unary (one variable) and clues 4-13 are binary (two variables).

Without any reasoning there are 4 x 4 x 4 x 3 = 192 combinations.

Most clues are IF-THEN rules. For example clue 7 is: IF Weapon = Trophy THEN Location = Library.
IF A THEN B is the same as (NOT A) OR B, so we wrote a small `implies()` function for it.

In [3]:
variables = ["Culprit", "Weapon", "Location", "Time"]

# list() makes a copy so the original lists are not changed
domains = {
    "Culprit":  list(SUSPECTS),
    "Weapon":   list(WEAPONS),
    "Location": list(LOCATIONS),
    "Time":     list(TIMES),
}

In [4]:
# IF a THEN b  is the same as  (not a) or b
def implies(a, b):
    return (not a) or b


# (variable, condition, clue)
unary_constraints = [
    ("Weapon",   lambda w: w != "Poison",     "Clue 1: No poison found in the body."),
    ("Location", lambda loc: loc != "Garden", "Clue 2: No footprints in the garden."),
    ("Time",     lambda t: t != "8 PM",       "Clue 3: Vikram was alive at 8 PM."),
]

# (variable 1, variable 2, condition, clue)
binary_constraints = [
    ("Weapon", "Location", lambda w, loc: implies(w == "Knife", loc == "Kitchen"),
     "Clue 4: No knife ever left the kitchen."),
    ("Culprit", "Location", lambda c, loc: implies(c == "Kabir", loc == "Kitchen"),
     "Clue 6: Kabir never left the kitchen."),
    ("Weapon", "Location", lambda w, loc: implies(w == "Trophy", loc == "Library"),
     "Clue 7: The trophy never left the library."),
    ("Location", "Time", lambda loc, t: implies(loc == "Library", t != "10 PM"),
     "Clue 8: The library was locked at 9:30 PM."),
    ("Location", "Time", lambda loc, t: implies(loc == "Balcony", t != "9 PM"),
     "Clue 9: The balcony was crowded at 9 PM."),
    ("Culprit", "Weapon", lambda c, w: implies(w == "Scarf", c in ["Anita", "Meera"]),
     "Clue 10: Only Anita and Meera could get the scarf."),
    ("Culprit", "Weapon", lambda c, w: implies(c == "Anita", w != "Trophy"),
     "Clue 11: Anita cannot lift the trophy."),
    ("Culprit", "Time", lambda c, t: implies(c == "Meera", t != "9 PM"),
     "Clue 12: Meera was on a phone call at 9 PM."),
    ("Culprit", "Time", lambda c, t: implies(c == "Anita", t != "10 PM"),
     "Clue 13: Anita left the party at 9:45 PM."),
]

In [5]:
# returns the clues broken by a (full or partial) assignment
# a clue is only checked when all of its variables have a value
def broken_clues(assignment):
    broken = []
    for var, rule, clue in unary_constraints:
        if var in assignment and not rule(assignment[var]):
            broken.append(clue)
    for var1, var2, rule, clue in binary_constraints:
        if var1 in assignment and var2 in assignment:
            if not rule(assignment[var1], assignment[var2]):
                broken.append(clue)
    return broken


def is_consistent(assignment):
    return len(broken_clues(assignment)) == 0


# hypothesis as a readable sentence
def describe(h):
    return f"{h['Culprit']} ({ROLES[h['Culprit']]}) with the {h['Weapon']} in the {h['Location']} at {h['Time']}"

## 3. Constraint Propagation

Before searching we remove values that can never be part of a solution.

1. **Node consistency:** remove the values that break a unary clue.
2. **AC-3 (arc consistency):** for an arc (X, Y), a value of X is removed if no value of Y works with it.
   When the domain of X changes, the arcs (Z, X) are added back to the queue. If a domain becomes empty,
   there is no solution.

Time complexity of AC-3 is O(c d³), where c is the number of constraints and d is the largest domain size.

In [6]:
def apply_unary_constraints(doms, verbose=False):
    new_doms = {var: list(values) for var, values in doms.items()}
    for var, rule, clue in unary_constraints:
        for value in list(new_doms[var]):
            if not rule(value):
                new_doms[var].remove(value)
                if verbose:
                    print(f"  removed {value} from {var}  ({clue})")
    return new_doms


# variables that share a binary constraint with var
def neighbours(var):
    result = []
    for var1, var2, rule, clue in binary_constraints:
        if var1 == var and var2 not in result:
            result.append(var2)
        if var2 == var and var1 not in result:
            result.append(var1)
    return result


# removes values of x that have no supporting value in y
def revise(doms, x, y, verbose=False):
    removed = False
    for vx in list(doms[x]):
        has_support = False
        for vy in doms[y]:
            if is_consistent({x: vx, y: vy}):
                has_support = True
                break
        if not has_support:
            doms[x].remove(vx)
            removed = True
            if verbose:
                print(f"  removed {vx} from {x}  (no value of {y} works with it)")
    return removed


def ac3(doms, verbose=False):
    doms = {var: list(values) for var, values in doms.items()}

    queue = []
    for var1, var2, rule, clue in binary_constraints:
        for arc in [(var1, var2), (var2, var1)]:
            if arc not in queue:
                queue.append(arc)

    while queue:
        x, y = queue.pop(0)
        if revise(doms, x, y, verbose):
            if len(doms[x]) == 0:
                return None   # no solution
            for z in neighbours(x):
                if z != y and (z, x) not in queue:
                    queue.append((z, x))
    return doms


def print_domains(doms):
    size = 1
    for var in variables:
        print(f"  {var:<9}: {doms[var]}")
        size = size * len(doms[var])
    print(f"  combinations left: {size}")

## 4. Backtracking Search

AC-3 only looks at two variables at a time, so it does not always give the final answer. Backtracking assigns
one variable at a time (Culprit, Weapon, Location, Time), checks the clues after every assignment and goes
back as soon as a clue is broken. We keep all the solutions, not just the first one, because the next parts
need to rank them. `nodes` counts how many values were tried.

In [7]:
def select_unassigned_variable(assignment):
    for var in variables:
        if var not in assignment:
            return var


def backtrack(assignment, doms, solutions, stats):
    if len(assignment) == len(variables):
        solutions.append(dict(assignment))
        return

    var = select_unassigned_variable(assignment)
    for value in doms[var]:
        stats["nodes"] += 1
        assignment[var] = value
        if is_consistent(assignment):
            backtrack(assignment, doms, solutions, stats)
        del assignment[var]   # undo and try the next value


def backtracking_search(doms):
    solutions = []
    stats = {"nodes": 0}
    backtrack({}, doms, solutions, stats)
    return solutions, stats["nodes"]

## 5. Solving the Case

The result is stored in `consistent_hypotheses`, which is used by the next sections.

In [8]:
print("Initial domains:")
print_domains(domains)

print("\nNode consistency:")
domains_after_unary = apply_unary_constraints(domains, verbose=True)
print_domains(domains_after_unary)

print("\nAC-3:")
domains_after_ac3 = ac3(domains_after_unary, verbose=True)

if domains_after_ac3 is None:
    print("The clues contradict each other, no solution.")
    consistent_hypotheses = []
else:
    print_domains(domains_after_ac3)
    print("\nBacktracking:")
    consistent_hypotheses, nodes = backtracking_search(domains_after_ac3)
    print("  nodes explored:", nodes)

print(f"\n{TOTAL_HYPOTHESES} hypotheses reduced to {len(consistent_hypotheses)}:")
for i, h in enumerate(consistent_hypotheses, start=1):
    print(f"  {i}. {describe(h)}")

Initial domains:
  Culprit  : ['Anita', 'Rohan', 'Meera', 'Kabir']
  Weapon   : ['Knife', 'Poison', 'Trophy', 'Scarf']
  Location : ['Library', 'Kitchen', 'Garden', 'Balcony']
  Time     : ['8 PM', '9 PM', '10 PM']
  combinations left: 192

Node consistency:
  removed Poison from Weapon  (Clue 1: No poison found in the body.)
  removed Garden from Location  (Clue 2: No footprints in the garden.)
  removed 8 PM from Time  (Clue 3: Vikram was alive at 8 PM.)
  Culprit  : ['Anita', 'Rohan', 'Meera', 'Kabir']
  Weapon   : ['Knife', 'Trophy', 'Scarf']
  Location : ['Library', 'Kitchen', 'Balcony']
  Time     : ['9 PM', '10 PM']
  combinations left: 72

AC-3:
  Culprit  : ['Anita', 'Rohan', 'Meera', 'Kabir']
  Weapon   : ['Knife', 'Trophy', 'Scarf']
  Location : ['Library', 'Kitchen', 'Balcony']
  Time     : ['9 PM', '10 PM']
  combinations left: 72

Backtracking:
  nodes explored: 67

192 hypotheses reduced to 11:
  1. Anita (business partner) with the Knife in the Kitchen at 9 PM
  2. Anit

`explain()` shows which clues rule out a hypothesis.

In [9]:
def explain(h):
    print(describe(h))
    clues = broken_clues(h)
    if len(clues) == 0:
        print("  possible, breaks no clue\n")
    else:
        print("  not possible because:")
        for clue in clues:
            print("   -", clue)
        print()


explain({"Culprit": "Kabir", "Weapon": "Knife", "Location": "Kitchen", "Time": "9 PM"})
explain({"Culprit": "Anita", "Weapon": "Trophy", "Location": "Library", "Time": "10 PM"})
explain(consistent_hypotheses[0])

Kabir (chef) with the Knife in the Kitchen at 9 PM
  possible, breaks no clue

Anita (business partner) with the Trophy in the Library at 10 PM
  not possible because:
   - Clue 8: The library was locked at 9:30 PM.
   - Clue 11: Anita cannot lift the trophy.
   - Clue 13: Anita left the party at 9:45 PM.

Anita (business partner) with the Knife in the Kitchen at 9 PM
  possible, breaks no clue



## 6. Logical Knowledge Base and Inference

Here the same confirmed clues are written as **propositional logic** sentences and stored in a knowledge base (KB).

- **Symbols:** one symbol per value, for example `C_Rohan` = "Rohan is the culprit", `W_Trophy`, `L_Library`, `T_9PM`.
  There are 4 + 4 + 4 + 3 = 15 symbols.
- **Sentences:** each variable has exactly one true value (e.g. exactly one culprit), plus the 13 clues.
  IF-THEN clues use `implies()`, for example clue 7 is `W_Trophy => L_Library`.
- **Inference:** truth-table model checking (TT-ENTAILS). KB entails a sentence α if α is true in every model where
  the KB is true. There are 2^15 = 32768 models, so we can check all of them.

With entailment the agent can answer questions like "Is Kabir innocent?" (yes, proven), "Is Rohan guilty?"
(not proven, not disproven). A hypothesis is kept only if the KB does **not** entail its negation, i.e. at least
one model of the KB agrees with it.

Model checking is sound and complete for propositional logic, but its time is O(2^n) for n symbols.

In [10]:
# one symbol per value, e.g. sym("Culprit", "Rohan") -> "C_Rohan", sym("Time", "9 PM") -> "T_9PM"
def sym(var, value):
    return var[0] + "_" + value.replace(" ", "")

SYMBOLS = [sym(var, value) for var in variables for value in domains[var]]


def exactly_one(var):
    return lambda m: sum(m[sym(var, v)] for v in domains[var]) == 1


# knowledge base: list of (name, sentence), a sentence is a function of the model
KB = [
    ("Exactly one culprit",  exactly_one("Culprit")),
    ("Exactly one weapon",   exactly_one("Weapon")),
    ("Exactly one location", exactly_one("Location")),
    ("Exactly one time",     exactly_one("Time")),
    ("Clue 1",  lambda m: not m["W_Poison"]),
    ("Clue 2",  lambda m: not m["L_Garden"]),
    ("Clue 3",  lambda m: not m["T_8PM"]),
    ("Clue 4",  lambda m: implies(m["W_Knife"], m["L_Kitchen"])),
    ("Clue 6",  lambda m: implies(m["C_Kabir"], m["L_Kitchen"])),
    ("Clue 7",  lambda m: implies(m["W_Trophy"], m["L_Library"])),
    ("Clue 8",  lambda m: implies(m["L_Library"], not m["T_10PM"])),
    ("Clue 9",  lambda m: implies(m["L_Balcony"], not m["T_9PM"])),
    ("Clue 10", lambda m: implies(m["W_Scarf"], m["C_Anita"] or m["C_Meera"])),
    ("Clue 11", lambda m: implies(m["C_Anita"], not m["W_Trophy"])),
    ("Clue 12", lambda m: implies(m["C_Meera"], not m["T_9PM"])),
    ("Clue 13", lambda m: implies(m["C_Anita"], not m["T_10PM"])),
]


def kb_true(m):
    return all(sentence(m) for name, sentence in KB)


# truth table: try every True/False combination of the symbols, keep the models of the KB
kb_models = []
for values in product([True, False], repeat=len(SYMBOLS)):
    m = dict(zip(SYMBOLS, values))
    if kb_true(m):
        kb_models.append(m)

print(f"models checked: {2 ** len(SYMBOLS)}, models of the KB: {len(kb_models)}")


# KB entails alpha if alpha is true in every model of the KB
def entails(alpha):
    return all(alpha(m) for m in kb_models)


# answers a yes / no question about one symbol
def ask(s):
    if entails(lambda m: m[s]):
        return "TRUE (proven)"
    if entails(lambda m: not m[s]):
        return "FALSE (proven)"
    return "UNKNOWN"

models checked: 32768, models of the KB: 11


In [11]:
print("What the KB can prove:")
for s in SYMBOLS:
    print(f"  {s:<11} {ask(s)}")

print("\nIs Kabir innocent?", entails(lambda m: not m["C_Kabir"]))
print("Is Rohan guilty?  ", entails(lambda m: m["C_Rohan"]))


# symbols that are true in a hypothesis
def h_symbols(h):
    return [sym(var, h[var]) for var in variables]


# keep h if the KB does not entail NOT h
logic_hypotheses = []
for h in consistent_hypotheses:
    hs = h_symbols(h)
    if not entails(lambda m: not all(m[s] for s in hs)):
        logic_hypotheses.append(h)

print(f"\n{len(consistent_hypotheses)} hypotheses from the CSP, {len(logic_hypotheses)} left after logical inference:")
for i, h in enumerate(logic_hypotheses, start=1):
    print(f"  {i}. {describe(h)}")

What the KB can prove:
  C_Anita     UNKNOWN
  C_Rohan     UNKNOWN
  C_Meera     UNKNOWN
  C_Kabir     UNKNOWN
  W_Knife     UNKNOWN
  W_Poison    FALSE (proven)
  W_Trophy    UNKNOWN
  W_Scarf     UNKNOWN
  L_Library   UNKNOWN
  L_Kitchen   UNKNOWN
  L_Garden    FALSE (proven)
  L_Balcony   UNKNOWN
  T_8PM       FALSE (proven)
  T_9PM       UNKNOWN
  T_10PM      UNKNOWN

Is Kabir innocent? False
Is Rohan guilty?   False

11 hypotheses from the CSP, 11 left after logical inference:
  1. Anita (business partner) with the Knife in the Kitchen at 9 PM
  2. Anita (business partner) with the Scarf in the Library at 9 PM
  3. Anita (business partner) with the Scarf in the Kitchen at 9 PM
  4. Rohan (nephew) with the Knife in the Kitchen at 9 PM
  5. Rohan (nephew) with the Knife in the Kitchen at 10 PM
  6. Rohan (nephew) with the Trophy in the Library at 9 PM
  7. Meera (secretary) with the Knife in the Kitchen at 10 PM
  8. Meera (secretary) with the Scarf in the Kitchen at 10 PM
  9. Meer

## 7. Bayesian Network and Hypothesis Ranking

**Input:** `logic_hypotheses` and the uncertain evidence defined in the Bayesian Network section.

**Output:** `ranked_hypotheses`, a list of (hypothesis, probability) pairs.

**Function:** The Bayesian Network contains one hypothesis node (H) representing the suspect, weapon, location and time combination. Four evidence nodes represent Motive, Fingerprint, Argument and Sighting.

The hypothesis node is the parent of all four evidence nodes. Each evidence node has a Conditional Probability Table (CPT) describing the likelihood of observing that evidence under a particular hypothesis.

Inference: Exact inference by enumeration is used to calculate the posterior probability of each hypothesis given the observed evidence.

The Bayesian model only ranks hypotheses that have already passed CSP and logical reasoning. It does not override hard logical constraints.

The conditional independence assumption is that the four evidence observations are independent given the complete hypothesis H.

In [12]:

# ============================================================
# BAYESIAN NETWORK: STRUCTURE, CPTs AND EXACT INFERENCE
# ============================================================

# 1. Network structure
# H represents the complete hypothesis:
# (Culprit, Weapon, Location, Time)

BN_NODES = [
    "H",
    "Motive",
    "Fingerprint",
    "Argument",
    "Sighting"
]

# Directed edges: H -> each evidence node
BN_EDGES = [
    ("H", "Motive"),
    ("H", "Fingerprint"),
    ("H", "Argument"),
    ("H", "Sighting")
]

EVIDENCE_NODES = ["Motive", "Fingerprint", "Argument", "Sighting"]


# 2. Prior probability of each feasible hypothesis
def uniform_prior(hyps):
    if len(hyps) == 0:
        return []
    return [1.0 / len(hyps)] * len(hyps)


# 3. Conditional Probability Tables (CPTs)
# Values are modelling assumptions, not measured frequencies.

EVIDENCE = {
    "Motive": {
        "text": "Rohan has gambling debts and gets money from the will",
        "p_true": lambda h: (
            0.80 if h["Culprit"] == "Rohan" else 0.30
        ),
    },

    "Fingerprint": {
        "text": "Partial fingerprint on the trophy, about 70% match with Rohan",
        "p_true": lambda h: (
            0.70
            if h["Culprit"] == "Rohan"
            and h["Weapon"] == "Trophy"
            else 0.15
        ),
    },

    "Argument": {
        "text": "Anita and Vikram argued in the morning",
        "p_true": lambda h: (
            0.60 if h["Culprit"] == "Anita" else 0.25
        ),
    },

    "Sighting": {
        "text": "A guest saw someone on the balcony around 10 PM",
        "p_true": lambda h: (
            0.60
            if h["Location"] == "Balcony"
            and h["Time"] == "10 PM"
            else 0.15
        ),
    },
}


# 4. Validate network structure and CPT probabilities
def validate_bn():
    assert ("H", "Motive") in BN_EDGES
    assert ("H", "Fingerprint") in BN_EDGES
    assert ("H", "Argument") in BN_EDGES
    assert ("H", "Sighting") in BN_EDGES

    assert set(EVIDENCE) == set(EVIDENCE_NODES)

    for h in logic_hypotheses:
        for name in EVIDENCE_NODES:
            p = EVIDENCE[name]["p_true"](h)
            assert 0.0 <= p <= 1.0

validate_bn()


# 5. CPT lookup
# Each evidence node has two outcomes: True and False.

def cpt_probability(h, node, value):
    p_true = EVIDENCE[node]["p_true"](h)
    return p_true if value else 1.0 - p_true


# 6. Joint probability using the Bayesian Network factorisation
#
# P(H, E1, E2, E3, E4)
# = P(H) * P(E1|H) * P(E2|H)
#       * P(E3|H) * P(E4|H)

def joint_probability(h, observed, prior=None):
    if prior is None:
        prior = uniform_prior(logic_hypotheses)

    index = logic_hypotheses.index(h)
    p = prior[index]

    for name, value in observed.items():
        if name not in EVIDENCE:
            raise ValueError(f"Unknown evidence node: {name}")

        p *= cpt_probability(h, name, value)

    return p


# 7. Exact posterior inference
#
# P(H | E) = P(H,E) / sum(P(H,E))

def posterior(hyps, observed, prior=None):
    if len(hyps) == 0:
        return []

    if prior is None:
        prior = uniform_prior(hyps)

    if len(prior) != len(hyps):
        raise ValueError("Prior length must match hypothesis count.")

    if any(p < 0 for p in prior):
        raise ValueError("Prior probabilities cannot be negative.")

    if abs(sum(prior) - 1.0) > 1e-9:
        raise ValueError("Prior probabilities must sum to 1.")

    joint = []

    for i, h in enumerate(hyps):
        p = prior[i]

        for name, value in observed.items():
            if name not in EVIDENCE:
                raise ValueError(f"Unknown evidence node: {name}")

            p *= cpt_probability(h, name, value)

        joint.append(p)

    total = sum(joint)

    if total <= 0:
        raise ValueError(
            "Observed evidence has zero probability under all hypotheses."
        )

    return [p / total for p in joint]


# 8. Rank hypotheses by posterior probability
def rank_hypotheses(hyps, observed, prior=None):
    probs = posterior(hyps, observed, prior)

    return sorted(
        zip(hyps, probs),
        key=lambda pair: pair[1],
        reverse=True
    )


# 9. Bayes factor for comparing two hypotheses
def bayes_factor(h, other, name, value=True):
    a = cpt_probability(h, name, value)
    b = cpt_probability(other, name, value)

    if b == 0:
        return float("inf") if a > 0 else 1.0

    return a / b


# 10. All four uncertain clues are initially reported
observed = {name: True for name in EVIDENCE_NODES}

ranked_hypotheses = rank_hypotheses(
    logic_hypotheses,
    observed
)


# 11. Display network structure
print("BAYESIAN NETWORK STRUCTURE")
print("Nodes:", BN_NODES)
print("Edges:", BN_EDGES)

print("\nCONDITIONAL PROBABILITY TABLES")

for h in logic_hypotheses:
    print("\nHypothesis:", describe(h))

    for name in EVIDENCE_NODES:
        p = EVIDENCE[name]["p_true"](h)
        print(f"  P({name}=True | H) = {p:.3f}")
        print(f"  P({name}=False | H) = {1-p:.3f}")


# 12. Show posterior updates as evidence arrives
print("\nPOSTERIOR UPDATES")

seen = {}

print(
    "No evidence:",
    [round(p, 3) for p in posterior(logic_hypotheses, seen)]
)

for name in EVIDENCE_NODES:
    seen[name] = True

    print(
        f"After {name}:",
        [round(p, 3) for p in posterior(logic_hypotheses, seen)]
    )


# 13. Final Bayesian ranking
print("\nFINAL BAYESIAN RANKING")

for h, p in ranked_hypotheses:
    print(f"P = {p:.4f} | {describe(h)}")


BAYESIAN NETWORK STRUCTURE
Nodes: ['H', 'Motive', 'Fingerprint', 'Argument', 'Sighting']
Edges: [('H', 'Motive'), ('H', 'Fingerprint'), ('H', 'Argument'), ('H', 'Sighting')]

CONDITIONAL PROBABILITY TABLES

Hypothesis: Anita (business partner) with the Knife in the Kitchen at 9 PM
  P(Motive=True | H) = 0.300
  P(Motive=False | H) = 0.700
  P(Fingerprint=True | H) = 0.150
  P(Fingerprint=False | H) = 0.850
  P(Argument=True | H) = 0.600
  P(Argument=False | H) = 0.400
  P(Sighting=True | H) = 0.150
  P(Sighting=False | H) = 0.850

Hypothesis: Anita (business partner) with the Scarf in the Library at 9 PM
  P(Motive=True | H) = 0.300
  P(Motive=False | H) = 0.700
  P(Fingerprint=True | H) = 0.150
  P(Fingerprint=False | H) = 0.850
  P(Argument=True | H) = 0.600
  P(Argument=False | H) = 0.400
  P(Sighting=True | H) = 0.150
  P(Sighting=False | H) = 0.850

Hypothesis: Anita (business partner) with the Scarf in the Kitchen at 9 PM
  P(Motive=True | H) = 0.300
  P(Motive=False | H) = 0.700

## 8. Temporal Reasoning (HMM)

**Input:** `CLUE_STREAM`, `START`, `TRANS`, and `EMIT`.

**Output:** State probabilities, the most likely hidden state sequence, and murder time probability distribution.

**Function:** Hidden states: Alive, Killed, Moved.

Observations: SeenAlive, Normal, Suspicious, BodyFound.

Algorithms:

Forward Algorithm: Calculates the probability distribution over the current hidden state after each observation.

Forward-Backward Algorithm: Calculates smoothed probabilities for each hidden state using the complete observation sequence.

Viterbi Algorithm: Identifies the single most probable hidden-state sequence.

Murder-Time Inference: Calculates the probability distribution of the first transition into the Killed state, restricted to murder times allowed by the CSP.

The HMM models how Vikram's condition changes over time and uses sequential observations to estimate the most likely progression of events.

In [13]:
# Hidden state = condition of Vikram at each step: Alive, Killed (body at the scene), Moved (body moved).
# Observation = the type of clue reported at that step.
# Steps are 8 PM, 9 PM, 10 PM and the next morning.

STATES = ["Alive", "Killed", "Moved"]
STEPS = ["8 PM", "9 PM", "10 PM", "Morning"]

# P(first state)
START = {"Alive": 1.0, "Killed": 0.0, "Moved": 0.0}      # clue 3: alive at 8 PM

# P(next state | state)  (one step = one clock hour, last step = overnight)
TRANS = {
    "Alive":  {"Alive": 0.50, "Killed": 0.50, "Moved": 0.00},
    "Killed": {"Alive": 0.00, "Killed": 0.40, "Moved": 0.60},
    "Moved":  {"Alive": 0.00, "Killed": 0.00, "Moved": 1.00},
}

# P(observation | state)
OBSERVATIONS = ["SeenAlive", "Normal", "Suspicious", "BodyFound"]
EMIT = {
    "Alive":  {"SeenAlive": 0.60, "Normal": 0.37, "Suspicious": 0.03, "BodyFound": 0.00},
    "Killed": {"SeenAlive": 0.00, "Normal": 0.45, "Suspicious": 0.50, "BodyFound": 0.05},
    "Moved":  {"SeenAlive": 0.00, "Normal": 0.55, "Suspicious": 0.20, "BodyFound": 0.25},
}
# clues in the order they arrive
CLUE_STREAM = ["SeenAlive", "Normal", "Suspicious", "BodyFound"]

def forward(obs, start=START, trans=TRANS, emit=EMIT):
    """filtering: belief over the current state after each clue (normalised)."""
    beliefs, alpha = [], {s: start[s] * emit[s][obs[0]] for s in STATES}
    for t in range(len(obs)):
        if t > 0:
            alpha = {s: sum(beliefs[-1][p] * trans[p][s] for p in STATES) * emit[s][obs[t]] for s in STATES}
        total = sum(alpha.values())
        beliefs.append({s: alpha[s] / total for s in STATES})
    return beliefs

def forward_backward(obs, start=START, trans=TRANS, emit=EMIT):
    """smoothing: belief over the state at every step using ALL clues."""
    n = len(obs)
    fwd = []
    a = {s: start[s] * emit[s][obs[0]] for s in STATES}
    fwd.append(a)
    for t in range(1, n):
        a = {s: sum(fwd[-1][p] * trans[p][s] for p in STATES) * emit[s][obs[t]] for s in STATES}
        fwd.append(a)
    bwd = [None] * n
    bwd[n - 1] = {s: 1.0 for s in STATES}
    for t in range(n - 2, -1, -1):
        bwd[t] = {s: sum(trans[s][q] * emit[q][obs[t + 1]] * bwd[t + 1][q] for q in STATES) for s in STATES}
    out = []
    for t in range(n):
        g = {s: fwd[t][s] * bwd[t][s] for s in STATES}
        z = sum(g.values())
        out.append({s: g[s] / z for s in STATES})
    return out

def viterbi(obs, start=START, trans=TRANS, emit=EMIT):
    """most likely hidden sequence."""
    v = [{s: start[s] * emit[s][obs[0]] for s in STATES}]
    back = []
    for t in range(1, len(obs)):
        v.append({}); back.append({})
        for s in STATES:
            best_prev = max(STATES, key=lambda p: v[t - 1][p] * trans[p][s])
            v[t][s] = v[t - 1][best_prev] * trans[best_prev][s] * emit[s][obs[t]]
            back[t - 1][s] = best_prev
    last = max(STATES, key=lambda s: v[-1][s])
    path = [last]
    for t in range(len(obs) - 2, -1, -1):
        path.insert(0, back[t][path[0]])
    return path, v[-1][last]

def murder_time_distribution(obs, allowed=("9 PM", "10 PM")):
    """P(murder at each time): sum the probability of every hidden path where the first
    Killed state is at that time. Only the times the CSP allows are kept, then renormalised."""
    n = len(obs); result = {}
    for k in range(1, n):          # 8 PM is impossible (clue 3: alive)
        if STEPS[k] not in allowed:
            continue
        # path prob = alive until k-1, killed at k, anything after
        p = START["Alive"] * EMIT["Alive"][obs[0]]
        for t in range(1, k):
            p *= TRANS["Alive"]["Alive"] * EMIT["Alive"][obs[t]]
        p *= TRANS["Alive"]["Killed"] * EMIT["Killed"][obs[k]]
        # sum over continuations from Killed at step k
        cont = {s: (1.0 if s == "Killed" else 0.0) for s in STATES}
        for t in range(k + 1, n):
            cont = {s: sum(cont[q] * TRANS[q][s] for q in STATES) * EMIT[s][obs[t]] for s in STATES}
        result[STEPS[k]] = p * sum(cont.values())
    z = sum(result.values())
    return {k: v / z for k, v in result.items()}



# ---- run it ----
obs = CLUE_STREAM
print("Clues:", list(zip(STEPS, obs)))

print("\nFiltering (belief after each new clue):")
for t, b in enumerate(forward(obs)):
    print(f"  {STEPS[t]:<8} {obs[t]:<11}", {s: round(p, 3) for s, p in b.items()})

print("\nSmoothing (belief at each step using all clues):")
for t, b in enumerate(forward_backward(obs)):
    print(f"  {STEPS[t]:<8}", {s: round(p, 3) for s, p in b.items()})

best_path, best_prob = viterbi(obs)
print("\nViterbi, most likely hidden sequence:")
for step, state in zip(STEPS, best_path):
    print(f"  {step:<8} {state}")

time_probs = murder_time_distribution(obs)
print("\nP(murder time):", {k: round(v, 3) for k, v in time_probs.items()})

Clues: [('8 PM', 'SeenAlive'), ('9 PM', 'Normal'), ('10 PM', 'Suspicious'), ('Morning', 'BodyFound')]

Filtering (belief after each new clue):
  8 PM     SeenAlive   {'Alive': 1.0, 'Killed': 0.0, 'Moved': 0.0}
  9 PM     Normal      {'Alive': 0.451, 'Killed': 0.549, 'Moved': 0.0}
  10 PM    Suspicious  {'Alive': 0.023, 'Killed': 0.754, 'Moved': 0.223}
  Morning  BodyFound   {'Alive': 0.0, 'Killed': 0.085, 'Moved': 0.915}

Smoothing (belief at each step using all clues):
  8 PM     {'Alive': 1.0, 'Killed': 0.0, 'Moved': 0.0}
  9 PM     {'Alive': 0.355, 'Killed': 0.645, 'Moved': 0.0}
  10 PM    {'Alive': 0.003, 'Killed': 0.695, 'Moved': 0.302}
  Morning  {'Alive': 0.0, 'Killed': 0.085, 'Moved': 0.915}

Viterbi, most likely hidden sequence:
  8 PM     Alive
  9 PM     Alive
  10 PM    Killed
  Morning  Moved

P(murder time): {'9 PM': 0.647, '10 PM': 0.353}


## 9. Streamlit Interface

An interactive web page for the AI Detective. It reuses the solver code from Sections 2-8 and 10 unchanged.

**Features**
- **Case overview:** the case, the confirmed clues (with a note for any clue the solver does not encode) and the uncertain evidence.
- **Solve Case:** runs CSP → logical inference → Bayesian ranking and shows the hypothesis counts. **Reset** clears it.
- **Verdict:** ranking table, probability bar chart, `verdict()`, and the evidence for and against each hypothesis.
- **Evidence controls:** set Motive, Fingerprint, Argument and Sighting to True, False or Unknown.
  Unknown leaves the item out of `observed`, it is not the same as False. `next_clue()` suggests what to check next.
- **Logic queries:** `ask()` gives *Proven true*, *Proven false* or *Unknown* for any symbol.
- **Hypothesis checker:** `broken_clues()` lists the constraints a theory breaks.
- **Timeline (HMM):** murder-time distribution, Viterbi path and state probabilities, kept separate from the Bayesian ranking.
- **Diagnostics:** domain reductions, ruled-out values and the solver log.

**Files.** Cells 9.1 and 9.2 write `app.py` and `requirements.txt`. These are the same files you upload to GitHub
for Streamlit Community Cloud. `app.py` has its own copy of the solver, so it runs without this notebook.

**Launching from Colab is optional.** Set `LAUNCH_APP = True` in cell 9.3, then run cells 9.3 to 9.6.
With the default `False`, *Run all* only writes the two files and skips the install and launch steps.

### 9.1 Generate `app.py`
Complete Streamlit app: the solver code copied from the notebook (no prints or tests), then wrappers and the interface.

In [ ]:
%%writefile app.py
"""
AI Detective: A Knowledge-Based Mystery Solving Agent
CSE518 Artificial Intelligence, Monsoon 2026 - Streamlit interface

Run locally:   streamlit run app.py
The solver code below is copied unchanged from the project notebook (Sections 1-8 and 10).
The interface code at the end only calls those functions through small wrappers.
"""
import hashlib
import io
import re
from contextlib import redirect_stdout
from pathlib import Path

import altair as alt
import pandas as pd
import streamlit as st

# =====================================================================
# SOLVER CODE (copied unchanged from the notebook, no prints or tests)
# =====================================================================
# ----- from notebook cell 1 -----
from itertools import product

# ----- from notebook cell 3 -----
SUSPECTS  = ["Anita", "Rohan", "Meera", "Kabir"]

WEAPONS   = ["Knife", "Poison", "Trophy", "Scarf"]

LOCATIONS = ["Library", "Kitchen", "Garden", "Balcony"]

TIMES     = ["8 PM", "9 PM", "10 PM"]

ROLES = {"Anita": "business partner", "Rohan": "nephew",
         "Meera": "secretary", "Kabir": "chef"}

TOTAL_HYPOTHESES = len(SUSPECTS) * len(WEAPONS) * len(LOCATIONS) * len(TIMES)

# ----- from notebook cell 5 -----
variables = ["Culprit", "Weapon", "Location", "Time"]

# list() makes a copy so the original lists are not changed
domains = {
    "Culprit":  list(SUSPECTS),
    "Weapon":   list(WEAPONS),
    "Location": list(LOCATIONS),
    "Time":     list(TIMES),
}

# ----- from notebook cell 6 -----
# IF a THEN b  is the same as  (not a) or b
def implies(a, b):
    return (not a) or b

# (variable, condition, clue)
unary_constraints = [
    ("Weapon",   lambda w: w != "Poison",     "Clue 1: No poison found in the body."),
    ("Location", lambda loc: loc != "Garden", "Clue 2: No footprints in the garden."),
    ("Time",     lambda t: t != "8 PM",       "Clue 3: Vikram was alive at 8 PM."),
]

# (variable 1, variable 2, condition, clue)
binary_constraints = [
    ("Weapon", "Location", lambda w, loc: implies(w == "Knife", loc == "Kitchen"),
     "Clue 4: No knife ever left the kitchen."),
    ("Culprit", "Location", lambda c, loc: implies(c == "Kabir", loc == "Kitchen"),
     "Clue 6: Kabir never left the kitchen."),
    ("Weapon", "Location", lambda w, loc: implies(w == "Trophy", loc == "Library"),
     "Clue 7: The trophy never left the library."),
    ("Location", "Time", lambda loc, t: implies(loc == "Library", t != "10 PM"),
     "Clue 8: The library was locked at 9:30 PM."),
    ("Location", "Time", lambda loc, t: implies(loc == "Balcony", t != "9 PM"),
     "Clue 9: The balcony was crowded at 9 PM."),
    ("Culprit", "Weapon", lambda c, w: implies(w == "Scarf", c in ["Anita", "Meera"]),
     "Clue 10: Only Anita and Meera could get the scarf."),
    ("Culprit", "Weapon", lambda c, w: implies(c == "Anita", w != "Trophy"),
     "Clue 11: Anita cannot lift the trophy."),
    ("Culprit", "Time", lambda c, t: implies(c == "Meera", t != "9 PM"),
     "Clue 12: Meera was on a phone call at 9 PM."),
    ("Culprit", "Time", lambda c, t: implies(c == "Anita", t != "10 PM"),
     "Clue 13: Anita left the party at 9:45 PM."),
]

# ----- from notebook cell 7 -----
# returns the clues broken by a (full or partial) assignment
# a clue is only checked when all of its variables have a value
def broken_clues(assignment):
    broken = []
    for var, rule, clue in unary_constraints:
        if var in assignment and not rule(assignment[var]):
            broken.append(clue)
    for var1, var2, rule, clue in binary_constraints:
        if var1 in assignment and var2 in assignment:
            if not rule(assignment[var1], assignment[var2]):
                broken.append(clue)
    return broken

def is_consistent(assignment):
    return len(broken_clues(assignment)) == 0

# hypothesis as a readable sentence
def describe(h):
    return f"{h['Culprit']} ({ROLES[h['Culprit']]}) with the {h['Weapon']} in the {h['Location']} at {h['Time']}"

# ----- from notebook cell 9 -----
def apply_unary_constraints(doms, verbose=False):
    new_doms = {var: list(values) for var, values in doms.items()}
    for var, rule, clue in unary_constraints:
        for value in list(new_doms[var]):
            if not rule(value):
                new_doms[var].remove(value)
                if verbose:
                    print(f"  removed {value} from {var}  ({clue})")
    return new_doms

# variables that share a binary constraint with var
def neighbours(var):
    result = []
    for var1, var2, rule, clue in binary_constraints:
        if var1 == var and var2 not in result:
            result.append(var2)
        if var2 == var and var1 not in result:
            result.append(var1)
    return result

# removes values of x that have no supporting value in y
def revise(doms, x, y, verbose=False):
    removed = False
    for vx in list(doms[x]):
        has_support = False
        for vy in doms[y]:
            if is_consistent({x: vx, y: vy}):
                has_support = True
                break
        if not has_support:
            doms[x].remove(vx)
            removed = True
            if verbose:
                print(f"  removed {vx} from {x}  (no value of {y} works with it)")
    return removed

def ac3(doms, verbose=False):
    doms = {var: list(values) for var, values in doms.items()}

    queue = []
    for var1, var2, rule, clue in binary_constraints:
        for arc in [(var1, var2), (var2, var1)]:
            if arc not in queue:
                queue.append(arc)

    while queue:
        x, y = queue.pop(0)
        if revise(doms, x, y, verbose):
            if len(doms[x]) == 0:
                return None   # no solution
            for z in neighbours(x):
                if z != y and (z, x) not in queue:
                    queue.append((z, x))
    return doms

def print_domains(doms):
    size = 1
    for var in variables:
        print(f"  {var:<9}: {doms[var]}")
        size = size * len(doms[var])
    print(f"  combinations left: {size}")

# ----- from notebook cell 11 -----
def select_unassigned_variable(assignment):
    for var in variables:
        if var not in assignment:
            return var

def backtrack(assignment, doms, solutions, stats):
    if len(assignment) == len(variables):
        solutions.append(dict(assignment))
        return

    var = select_unassigned_variable(assignment)
    for value in doms[var]:
        stats["nodes"] += 1
        assignment[var] = value
        if is_consistent(assignment):
            backtrack(assignment, doms, solutions, stats)
        del assignment[var]   # undo and try the next value

def backtracking_search(doms):
    solutions = []
    stats = {"nodes": 0}
    backtrack({}, doms, solutions, stats)
    return solutions, stats["nodes"]

# ----- from notebook cell 17 -----
# one symbol per value, e.g. sym("Culprit", "Rohan") -> "C_Rohan", sym("Time", "9 PM") -> "T_9PM"
def sym(var, value):
    return var[0] + "_" + value.replace(" ", "")

SYMBOLS = [sym(var, value) for var in variables for value in domains[var]]

def exactly_one(var):
    return lambda m: sum(m[sym(var, v)] for v in domains[var]) == 1

# knowledge base: list of (name, sentence), a sentence is a function of the model
KB = [
    ("Exactly one culprit",  exactly_one("Culprit")),
    ("Exactly one weapon",   exactly_one("Weapon")),
    ("Exactly one location", exactly_one("Location")),
    ("Exactly one time",     exactly_one("Time")),
    ("Clue 1",  lambda m: not m["W_Poison"]),
    ("Clue 2",  lambda m: not m["L_Garden"]),
    ("Clue 3",  lambda m: not m["T_8PM"]),
    ("Clue 4",  lambda m: implies(m["W_Knife"], m["L_Kitchen"])),
    ("Clue 6",  lambda m: implies(m["C_Kabir"], m["L_Kitchen"])),
    ("Clue 7",  lambda m: implies(m["W_Trophy"], m["L_Library"])),
    ("Clue 8",  lambda m: implies(m["L_Library"], not m["T_10PM"])),
    ("Clue 9",  lambda m: implies(m["L_Balcony"], not m["T_9PM"])),
    ("Clue 10", lambda m: implies(m["W_Scarf"], m["C_Anita"] or m["C_Meera"])),
    ("Clue 11", lambda m: implies(m["C_Anita"], not m["W_Trophy"])),
    ("Clue 12", lambda m: implies(m["C_Meera"], not m["T_9PM"])),
    ("Clue 13", lambda m: implies(m["C_Anita"], not m["T_10PM"])),
]

def kb_true(m):
    return all(sentence(m) for name, sentence in KB)

# KB entails alpha if alpha is true in every model of the KB
def entails(alpha):
    return all(alpha(m) for m in kb_models)

# answers a yes / no question about one symbol
def ask(s):
    if entails(lambda m: m[s]):
        return "TRUE (proven)"
    if entails(lambda m: not m[s]):
        return "FALSE (proven)"
    return "UNKNOWN"

# ----- from notebook cell 18 -----
# symbols that are true in a hypothesis
def h_symbols(h):
    return [sym(var, h[var]) for var in variables]

# ----- from notebook cell 20 -----
BN_NODES = [
    "H",
    "Motive",
    "Fingerprint",
    "Argument",
    "Sighting"
]

# Directed edges: H -> each evidence node
BN_EDGES = [
    ("H", "Motive"),
    ("H", "Fingerprint"),
    ("H", "Argument"),
    ("H", "Sighting")
]

EVIDENCE_NODES = ["Motive", "Fingerprint", "Argument", "Sighting"]

# 2. Prior probability of each feasible hypothesis
def uniform_prior(hyps):
    if len(hyps) == 0:
        return []
    return [1.0 / len(hyps)] * len(hyps)

EVIDENCE = {
    "Motive": {
        "text": "Rohan has gambling debts and gets money from the will",
        "p_true": lambda h: (
            0.80 if h["Culprit"] == "Rohan" else 0.30
        ),
    },

    "Fingerprint": {
        "text": "Partial fingerprint on the trophy, about 70% match with Rohan",
        "p_true": lambda h: (
            0.70
            if h["Culprit"] == "Rohan"
            and h["Weapon"] == "Trophy"
            else 0.15
        ),
    },

    "Argument": {
        "text": "Anita and Vikram argued in the morning",
        "p_true": lambda h: (
            0.60 if h["Culprit"] == "Anita" else 0.25
        ),
    },

    "Sighting": {
        "text": "A guest saw someone on the balcony around 10 PM",
        "p_true": lambda h: (
            0.60
            if h["Location"] == "Balcony"
            and h["Time"] == "10 PM"
            else 0.15
        ),
    },
}

# 4. Validate network structure and CPT probabilities
def validate_bn():
    assert ("H", "Motive") in BN_EDGES
    assert ("H", "Fingerprint") in BN_EDGES
    assert ("H", "Argument") in BN_EDGES
    assert ("H", "Sighting") in BN_EDGES

    assert set(EVIDENCE) == set(EVIDENCE_NODES)

    for h in logic_hypotheses:
        for name in EVIDENCE_NODES:
            p = EVIDENCE[name]["p_true"](h)
            assert 0.0 <= p <= 1.0

def cpt_probability(h, node, value):
    p_true = EVIDENCE[node]["p_true"](h)
    return p_true if value else 1.0 - p_true

def joint_probability(h, observed, prior=None):
    if prior is None:
        prior = uniform_prior(logic_hypotheses)

    index = logic_hypotheses.index(h)
    p = prior[index]

    for name, value in observed.items():
        if name not in EVIDENCE:
            raise ValueError(f"Unknown evidence node: {name}")

        p *= cpt_probability(h, name, value)

    return p

def posterior(hyps, observed, prior=None):
    if len(hyps) == 0:
        return []

    if prior is None:
        prior = uniform_prior(hyps)

    if len(prior) != len(hyps):
        raise ValueError("Prior length must match hypothesis count.")

    if any(p < 0 for p in prior):
        raise ValueError("Prior probabilities cannot be negative.")

    if abs(sum(prior) - 1.0) > 1e-9:
        raise ValueError("Prior probabilities must sum to 1.")

    joint = []

    for i, h in enumerate(hyps):
        p = prior[i]

        for name, value in observed.items():
            if name not in EVIDENCE:
                raise ValueError(f"Unknown evidence node: {name}")

            p *= cpt_probability(h, name, value)

        joint.append(p)

    total = sum(joint)

    if total <= 0:
        raise ValueError(
            "Observed evidence has zero probability under all hypotheses."
        )

    return [p / total for p in joint]

# 8. Rank hypotheses by posterior probability
def rank_hypotheses(hyps, observed, prior=None):
    probs = posterior(hyps, observed, prior)

    return sorted(
        zip(hyps, probs),
        key=lambda pair: pair[1],
        reverse=True
    )

# 9. Bayes factor for comparing two hypotheses
def bayes_factor(h, other, name, value=True):
    a = cpt_probability(h, name, value)
    b = cpt_probability(other, name, value)

    if b == 0:
        return float("inf") if a > 0 else 1.0

    return a / b

# 10. All four uncertain clues are initially reported
observed = {name: True for name in EVIDENCE_NODES}

# ----- from notebook cell 22 -----
STATES = ["Alive", "Killed", "Moved"]

STEPS = ["8 PM", "9 PM", "10 PM", "Morning"]

# P(first state)
START = {"Alive": 1.0, "Killed": 0.0, "Moved": 0.0}      # clue 3: alive at 8 PM

# P(next state | state)  (one step = one clock hour, last step = overnight)
TRANS = {
    "Alive":  {"Alive": 0.50, "Killed": 0.50, "Moved": 0.00},
    "Killed": {"Alive": 0.00, "Killed": 0.40, "Moved": 0.60},
    "Moved":  {"Alive": 0.00, "Killed": 0.00, "Moved": 1.00},
}

# P(observation | state)
OBSERVATIONS = ["SeenAlive", "Normal", "Suspicious", "BodyFound"]

EMIT = {
    "Alive":  {"SeenAlive": 0.60, "Normal": 0.37, "Suspicious": 0.03, "BodyFound": 0.00},
    "Killed": {"SeenAlive": 0.00, "Normal": 0.45, "Suspicious": 0.50, "BodyFound": 0.05},
    "Moved":  {"SeenAlive": 0.00, "Normal": 0.55, "Suspicious": 0.20, "BodyFound": 0.25},
}

# clues in the order they arrive
CLUE_STREAM = ["SeenAlive", "Normal", "Suspicious", "BodyFound"]

def forward(obs, start=START, trans=TRANS, emit=EMIT):
    """filtering: belief over the current state after each clue (normalised)."""
    beliefs, alpha = [], {s: start[s] * emit[s][obs[0]] for s in STATES}
    for t in range(len(obs)):
        if t > 0:
            alpha = {s: sum(beliefs[-1][p] * trans[p][s] for p in STATES) * emit[s][obs[t]] for s in STATES}
        total = sum(alpha.values())
        beliefs.append({s: alpha[s] / total for s in STATES})
    return beliefs

def forward_backward(obs, start=START, trans=TRANS, emit=EMIT):
    """smoothing: belief over the state at every step using ALL clues."""
    n = len(obs)
    fwd = []
    a = {s: start[s] * emit[s][obs[0]] for s in STATES}
    fwd.append(a)
    for t in range(1, n):
        a = {s: sum(fwd[-1][p] * trans[p][s] for p in STATES) * emit[s][obs[t]] for s in STATES}
        fwd.append(a)
    bwd = [None] * n
    bwd[n - 1] = {s: 1.0 for s in STATES}
    for t in range(n - 2, -1, -1):
        bwd[t] = {s: sum(trans[s][q] * emit[q][obs[t + 1]] * bwd[t + 1][q] for q in STATES) for s in STATES}
    out = []
    for t in range(n):
        g = {s: fwd[t][s] * bwd[t][s] for s in STATES}
        z = sum(g.values())
        out.append({s: g[s] / z for s in STATES})
    return out

def viterbi(obs, start=START, trans=TRANS, emit=EMIT):
    """most likely hidden sequence."""
    v = [{s: start[s] * emit[s][obs[0]] for s in STATES}]
    back = []
    for t in range(1, len(obs)):
        v.append({}); back.append({})
        for s in STATES:
            best_prev = max(STATES, key=lambda p: v[t - 1][p] * trans[p][s])
            v[t][s] = v[t - 1][best_prev] * trans[best_prev][s] * emit[s][obs[t]]
            back[t - 1][s] = best_prev
    last = max(STATES, key=lambda s: v[-1][s])
    path = [last]
    for t in range(len(obs) - 2, -1, -1):
        path.insert(0, back[t][path[0]])
    return path, v[-1][last]

def murder_time_distribution(obs, allowed=("9 PM", "10 PM")):
    """P(murder at each time): sum the probability of every hidden path where the first
    Killed state is at that time. Only the times the CSP allows are kept, then renormalised."""
    n = len(obs); result = {}
    for k in range(1, n):          # 8 PM is impossible (clue 3: alive)
        if STEPS[k] not in allowed:
            continue
        # path prob = alive until k-1, killed at k, anything after
        p = START["Alive"] * EMIT["Alive"][obs[0]]
        for t in range(1, k):
            p *= TRANS["Alive"]["Alive"] * EMIT["Alive"][obs[t]]
        p *= TRANS["Alive"]["Killed"] * EMIT["Killed"][obs[k]]
        # sum over continuations from Killed at step k
        cont = {s: (1.0 if s == "Killed" else 0.0) for s in STATES}
        for t in range(k + 1, n):
            cont = {s: sum(cont[q] * TRANS[q][s] for q in STATES) * EMIT[s][obs[t]] for s in STATES}
        result[STEPS[k]] = p * sum(cont.values())
    z = sum(result.values())
    return {k: v / z for k, v in result.items()}

# ----- from notebook cell 26 -----
from math import log2

# ---------- 1. Bayesian Network posterior ranking ----------
# Uses exact inference from the explicitly defined Bayesian Network.
# For each observed clue, compare how likely the clue is under this hypothesis
# with how likely it is under the other hypotheses (ratio > 1 supports, < 1 conflicts).
SUPPORT_LIMIT = 1.2      # ratio above this counts as supporting

CONFLICT_LIMIT = 0.8     # ratio below this counts as conflicting

def clue_ratio(h, others, name, value):
    p_h = EVIDENCE[name]["p_true"](h)
    p_o = sum(EVIDENCE[name]["p_true"](o) for o in others) / len(others)
    if not value:
        p_h, p_o = 1 - p_h, 1 - p_o
    return p_h / p_o

def build_ranking(hyps, observed, clue_stream=None):
    """Returns a list of dicts (best first) with everything the interface needs."""
    probs = posterior(hyps, observed)
    time_probs = murder_time_distribution(clue_stream) if clue_stream else {}
    rows = []
    for i, h in enumerate(hyps):
        others = [o for j, o in enumerate(hyps) if j != i]
        supports, conflicts = [], []
        if others:
            for name, value in observed.items():
                ratio = clue_ratio(h, others, name, value)
                if ratio > SUPPORT_LIMIT:
                    supports.append((name, ratio))
                elif ratio < CONFLICT_LIMIT:
                    conflicts.append((name, ratio))
        rows.append({
            "hypothesis": h,
            "probability": probs[i],
            "supports": sorted(supports, key=lambda x: -x[1]),
            "conflicts": sorted(conflicts, key=lambda x: x[1]),
            "hmm_time_prob": time_probs.get(h["Time"]),
        })
    return sorted(rows, key=lambda r: r["probability"], reverse=True)

# ---------- 2. confidence: say when the evidence is not enough ----------
STRONG_TOP, STRONG_MARGIN = 0.80, 0.30

LIKELY_TOP, LIKELY_MARGIN = 0.50, 0.20

def verdict(rows):
    if len(rows) == 0:
        return "NO HYPOTHESIS", "The clues contradict each other."
    if len(rows) == 1:
        return "CONCLUSIVE", "Only one hypothesis is consistent with the clues."
    top, second = rows[0]["probability"], rows[1]["probability"]
    margin = top - second
    if top >= STRONG_TOP and margin >= STRONG_MARGIN:
        return "STRONG", f"Top hypothesis leads by {margin:.2f}."
    if top >= LIKELY_TOP and margin >= LIKELY_MARGIN:
        return "LIKELY", f"Top hypothesis leads by {margin:.2f}, but it is not conclusive."
    return "INCONCLUSIVE", f"Only {margin:.2f} between the top two, more evidence is needed."

# ---------- 3. decision layer: which clue should we look for next? ----------
def entropy(probs):
    return -sum(p * log2(p) for p in probs if p > 0)

def information_gain(hyps, observed, name):
    """Expected drop in entropy if we find out whether clue `name` is True or False."""
    now = posterior(hyps, observed)
    expected = 0.0
    for value in (True, False):
        p_value = sum(now[i] * (EVIDENCE[name]["p_true"](h) if value else 1 - EVIDENCE[name]["p_true"](h))
                      for i, h in enumerate(hyps))
        if p_value > 0:
            after = posterior(hyps, {**observed, name: value})
            expected += p_value * entropy(after)
    return entropy(now) - expected

def next_clue(hyps, observed):
    """The not-yet-known clue with the highest information gain (None if all are known)."""
    options = [(name, information_gain(hyps, observed, name)) for name in EVIDENCE if name not in observed]
    if len(options) == 0:
        return None
    return sorted(options, key=lambda x: -x[1])


# =====================================================================
# WRAPPERS FOR THE INTERFACE (new code, the solver functions are not changed)
# =====================================================================
EVIDENCE_CHOICES = ["True", "False", "Unknown"]
DEFAULT_OBSERVED = dict(observed)          # notebook default: all four items reported True
ASK_LABELS = {"TRUE (proven)": "Proven true", "FALSE (proven)": "Proven false", "UNKNOWN": "Unknown"}
CASE_CLUES = [
    "The autopsy found no poison in the body.",
    "It rained all evening and there were no footprints in the garden.",
    "Vikram gave a speech to all the guests at 8 PM, so he was alive at 8 PM.",
    "No knife ever left the kitchen.",
    "Kitchen helpers were in the kitchen all evening except at 8 PM, when they went to hear the speech.",
    "CCTV shows that Kabir never left the kitchen.",
    "The trophy is heavy and never left the library.",
    "The library was locked at 9:30 PM.",
    "At 9 PM a lot of guests were on the balcony watching fireworks.",
    "The scarf was kept in the office and only Anita and Meera had the office key.",
    "Anita has a broken wrist and cannot lift the trophy.",
    "Meera was on a recorded phone call at 9 PM.",
    "Anita left the party at 9:45 PM.",
]
ENCODED_CLUES = [c for _, _, c in unary_constraints] + [c for _, _, _, c in binary_constraints]
ENCODED_NUMBERS = sorted(int(re.search(r"Clue (\d+)", c).group(1)) for c in ENCODED_CLUES)
NOT_ENCODED = [n for n in range(1, len(CASE_CLUES) + 1) if n not in ENCODED_NUMBERS]


@st.cache_data(show_spinner=False)
def load_kb_models(source_version):
    """Same truth-table loop as the notebook. Returns plain dicts, so it is safe to share.
    source_version (a hash of this file) makes the cache refresh if the KB code is edited."""
    models = []
    for values in product([True, False], repeat=len(SYMBOLS)):
        m = dict(zip(SYMBOLS, values))
        if kb_true(m):
            models.append(m)
    return models


kb_models = load_kb_models(hashlib.sha256(Path(__file__).read_bytes()).hexdigest())   # used by entails() and ask()


def domain_combinations(doms):
    size = 1
    for var in variables:
        size = size * len(doms[var])
    return size


def logic_filter(hyps):
    """Same rule as the notebook: keep h if the KB does not entail NOT h."""
    kept = []
    for h in hyps:
        hs = h_symbols(h)
        if not entails(lambda m: not all(m[s] for s in hs)):
            kept.append(h)
    return kept


def cpts_valid(hyps):
    """Runs the notebook's validate_bn() on these hypotheses (global is per script run)."""
    global logic_hypotheses
    logic_hypotheses = hyps
    try:
        validate_bn()
        return True
    except AssertionError:
        return False


def solve_case():
    """CSP -> logical inference. Returns only plain data for st.session_state."""
    log = io.StringIO()
    with redirect_stdout(log):
        print("Initial domains:")
        print_domains(domains)
        print("\nNode consistency:")
        d_unary = apply_unary_constraints(domains, verbose=True)
        print_domains(d_unary)
        print("\nAC-3:")
        d_ac3 = ac3(d_unary, verbose=True)
        if d_ac3 is None:
            print("AC-3 emptied a domain, so the clues contradict each other.")
            csp, nodes = [], 0
        else:
            print_domains(d_ac3)
            csp, nodes = backtracking_search(d_ac3)
            print("\nBacktracking nodes explored:", nodes)
    kb_ok = len(kb_models) > 0
    logic = logic_filter(csp) if kb_ok else []
    return {
        "initial": {v: list(domains[v]) for v in variables},
        "unary": d_unary,
        "ac3": d_ac3,
        "csp": csp,
        "nodes": nodes,
        "kb_ok": kb_ok,
        "kb_count": len(kb_models),
        "logic": logic,
        "cpt_ok": cpts_valid(logic) if logic else True,
        "log": log.getvalue(),
    }


def default_choice(name):
    if name not in DEFAULT_OBSERVED:
        return "Unknown"
    return "True" if DEFAULT_OBSERVED[name] else "False"


def current_observed():
    """Unknown means the item is left out of the dictionary (it is not False)."""
    obs = {}
    for name in EVIDENCE_NODES:
        choice = st.session_state.get(f"ev_{name}", default_choice(name))
        if choice == "True":
            obs[name] = True
        elif choice == "False":
            obs[name] = False
    return obs


def safe_ranking(hyps, obs):
    if len(hyps) == 0:
        return [], None
    try:
        return build_ranking(hyps, obs), None
    except (ValueError, ZeroDivisionError) as e:
        return [], str(e)


def safe_next_clue(hyps, obs):
    if len(hyps) == 0:
        return None, None
    try:
        return next_clue(hyps, obs), None
    except (ValueError, ZeroDivisionError) as e:
        return None, str(e)


def safe_hmm():
    try:
        path, path_prob = viterbi(CLUE_STREAM)
        return {
            "time_probs": murder_time_distribution(CLUE_STREAM),
            "path": path,
            "path_prob": path_prob,
            "smooth": forward_backward(CLUE_STREAM),
            "filter": forward(CLUE_STREAM),
        }, None
    except (ValueError, ZeroDivisionError, KeyError) as e:
        return None, str(e)


def label(h, rank=None):
    text = f"{h['Culprit']} · {h['Weapon']} · {h['Location']} · {h['Time']}"
    return f"#{rank} {text}" if rank is not None else text


def proven_by_logic(h):
    """True only if the KB entails every part of h."""
    return len(kb_models) > 0 and all(ask(s) == "TRUE (proven)" for s in h_symbols(h))


def reset_all():
    st.session_state.pop("results", None)
    st.session_state.pop("check_result", None)
    for name in EVIDENCE_NODES:
        st.session_state[f"ev_{name}"] = default_choice(name)


# =====================================================================
# PAGE
# =====================================================================
st.set_page_config(page_title="AI Detective", page_icon="🔍", layout="centered")

st.markdown(
    """
    <style>
    h1 { letter-spacing: 0.5px; }
    div[data-testid="stMetricValue"] { font-size: 1.6rem; }
    </style>
    """,
    unsafe_allow_html=True,
)

for _name in EVIDENCE_NODES:
    if f"ev_{_name}" not in st.session_state:
        st.session_state[f"ev_{_name}"] = default_choice(_name)

st.title("🔍 AI Detective")
st.markdown(
    "**Who killed Vikram Shah at Riverside Manor?** The agent removes every possibility that breaks a clue "
    "(CSP + logic), then ranks the rest with a Bayesian network."
)
st.caption(
    "All deductions follow the puzzle's encoded rules and modelling assumptions. "
    "They are not real-world facts."
)

# ---------------- 1. Case overview ----------------
with st.expander("📜 The case"):
    st.markdown(
        "Vikram Shah was killed on the night of his 60th birthday party at Riverside Manor. "
        "His body was found the next morning near the staircase, but it had been moved there after the crime.\n\n"
        f"- **Suspects:** " + ", ".join(f"{s} ({ROLES[s]})" for s in SUSPECTS) + "\n"
        f"- **Weapons:** {', '.join(WEAPONS)}\n"
        f"- **Locations:** {', '.join(LOCATIONS)}\n"
        f"- **Times:** {', '.join(TIMES)}"
    )

with st.expander("✅ Confirmed clues"):
    for i, clue in enumerate(CASE_CLUES, start=1):
        tag = "" if i in ENCODED_NUMBERS else "  *(not encoded in the solver)*"
        st.markdown(f"{i}. {clue}{tag}")
    st.caption(f"{len(ENCODED_CLUES)} clues are encoded as constraints in the current solver.")
    if NOT_ENCODED:
        st.info(
            "Clue " + ", ".join(str(n) for n in NOT_ENCODED) +
            " is in the case text but is not one of the solver's constraints, so it does not affect the results."
        )

with st.expander("❓ Uncertain evidence (used only by the Bayesian network)"):
    for name in EVIDENCE_NODES:
        st.markdown(f"- **{name}:** {EVIDENCE[name]['text']}")
    st.caption("These items are not confirmed facts. They change probabilities, never the hard constraints.")

# ---------------- 2. Solve workflow ----------------
c1, c2 = st.columns(2)
if c1.button("Solve Case", type="primary", width="stretch"):
    with st.spinner("Reasoning through the clues..."):
        st.session_state["results"] = solve_case()
c2.button("Reset", on_click=reset_all, width="stretch")

res = st.session_state.get("results")

if res is None:
    st.info("Press **Solve Case** to run CSP → logical inference → Bayesian ranking.")
else:
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Initial hypotheses", TOTAL_HYPOTHESES)
    m2.metric("Domain combinations after AC-3", domain_combinations(res["ac3"]) if res["ac3"] else 0)
    m3.metric("CSP solutions", len(res["csp"]))
    m4.metric("After logical inference", len(res["logic"]))
    st.caption(
        "Domain combinations are only an upper bound (AC-3 checks two variables at a time). "
        "CSP solutions are complete hypotheses that break no encoded clue."
    )
    if res["ac3"] is None:
        st.error("AC-3 emptied a domain: the encoded clues contradict each other. There is no valid hypothesis.")
    elif len(res["csp"]) == 0:
        st.error("Backtracking found no complete hypothesis that satisfies every clue.")
    if not res["kb_ok"]:
        st.error("Inconsistent knowledge base: the KB has no satisfying model.")

tab_rank, tab_logic, tab_check, tab_hmm, tab_diag = st.tabs(
    ["Verdict", "Logic queries", "Check a hypothesis", "Timeline (HMM)", "Diagnostics"]
)

# ---------------- 3 + 4. Results, verdict, evidence controls ----------------
with tab_rank:
    st.subheader("Uncertain evidence")
    st.caption("Set each item to True, False or Unknown. **Unknown** means it is not observed and is left out; "
               "it does not mean False. Defaults are the notebook's observed evidence.")
    cols = st.columns(2)
    for i, name in enumerate(EVIDENCE_NODES):
        with cols[i % 2]:
            st.radio(f"**{name}**", EVIDENCE_CHOICES, key=f"ev_{name}", horizontal=True,
                     help=EVIDENCE[name]["text"])

    obs = current_observed()

    if res is None:
        st.info("Solve the case first to see the ranking.")
    elif len(res["logic"]) == 0:
        st.warning("No hypothesis is left after CSP and logic, so there is nothing to rank.")
    else:
        rows, err = safe_ranking(res["logic"], obs)
        if err:
            st.error(f"The ranking could not be computed with this evidence: {err}")
        if rows:
            if not res["cpt_ok"]:
                st.warning("A CPT value is outside [0, 1]; treat these probabilities with care.")
            st.subheader("Ranking")
            df = pd.DataFrame([{
                "Rank": i,
                "Culprit": r["hypothesis"]["Culprit"],
                "Weapon": r["hypothesis"]["Weapon"],
                "Location": r["hypothesis"]["Location"],
                "Time": r["hypothesis"]["Time"],
                "Posterior": r["probability"],
            } for i, r in enumerate(rows, start=1)])
            st.dataframe(
                df, hide_index=True, width="stretch",
                column_config={"Posterior": st.column_config.ProgressColumn(
                    "Posterior", format="%.3f", min_value=0.0, max_value=1.0)},
            )
            st.caption(f"Sum of posteriors = {sum(r['probability'] for r in rows):.3f}")

            chart_df = pd.DataFrame({
                "Hypothesis": [label(r["hypothesis"], i) for i, r in enumerate(rows, start=1)],
                "Posterior": [r["probability"] for r in rows],
            })
            chart = (
                alt.Chart(chart_df)
                .mark_bar()
                .encode(
                    x=alt.X("Posterior:Q", scale=alt.Scale(domain=[0, 1]), title="Posterior probability"),
                    y=alt.Y("Hypothesis:N", sort=None, title=None, axis=alt.Axis(labelLimit=260)),
                    tooltip=["Hypothesis", alt.Tooltip("Posterior:Q", format=".3f")],
                )
                .properties(height=28 * len(chart_df) + 40)
            )
            st.altair_chart(chart, width="stretch")

            # verdict
            st.subheader("Verdict")
            v_label, v_reason = verdict(rows)
            top = rows[0]
            tied = [r for r in rows if abs(r["probability"] - top["probability"]) < 1e-12]
            if len(tied) > 1:
                st.warning(f"**INCONCLUSIVE (tie).** {len(tied)} hypotheses share the top probability "
                           f"P = {top['probability']:.3f}, so no single answer can be picked.")
                for r in tied:
                    st.markdown(f"- {describe(r['hypothesis'])}")
            else:
                show = {"STRONG": st.success, "LIKELY": st.info, "CONCLUSIVE": st.success}.get(v_label, st.warning)
                show(f"**{v_label}.** {v_reason}")
                if proven_by_logic(top["hypothesis"]):
                    st.markdown(f"**Logically determined:** {describe(top['hypothesis'])}")
                else:
                    st.markdown(f"**Most probable (not proven):** {describe(top['hypothesis'])} "
                                f"(P = {top['probability']:.2f})")
            st.caption(
                "Probabilities depend on the assumed CPT values in the notebook and are conditional on the "
                f"{len(rows)} hypotheses that survived CSP and logic. Only logic can prove something; "
                "a high probability is not a proof."
            )

            with st.expander("Supporting and conflicting evidence"):
                st.caption("Ratio = how likely the observed item is under this hypothesis compared with the "
                           f"average of the others. Above {SUPPORT_LIMIT} supports, below {CONFLICT_LIMIT} conflicts.")
                if len(obs) == 0:
                    st.markdown("No evidence is observed, so nothing supports or conflicts yet.")
                for i, r in enumerate(rows, start=1):
                    sup = ", ".join(f"{n} (×{x:.1f})" for n, x in r["supports"]) or "none"
                    con = ", ".join(f"{n} (×{x:.1f})" for n, x in r["conflicts"]) or "none"
                    st.markdown(f"**{label(r['hypothesis'], i)}** · P = {r['probability']:.3f}  \n"
                                f"Supported by: {sup}  \nConflicts with: {con}")

        st.subheader("Which evidence to check next")
        options, err2 = safe_next_clue(res["logic"], obs)
        if err2:
            st.error(f"Information gain could not be computed: {err2}")
        elif options is None:
            st.markdown("Every evidence item is set to True or False, so no unobserved evidence items remain.")
        else:
            best_name, best_gain = options[0]
            st.markdown(f"Find out **{best_name}** next: expected information gain {best_gain:.3f} bits.")
            st.dataframe(pd.DataFrame(options, columns=["Evidence item", "Information gain (bits)"]),
                         hide_index=True, width="stretch")

# ---------------- 5. Logical queries ----------------
with tab_logic:
    st.subheader("Ask the knowledge base")
    if len(kb_models) == 0:
        st.error("Inconsistent knowledge base: no model satisfies it, so no truth claims are shown.")
    else:
        q1, q2 = st.columns(2)
        q_var = q1.selectbox("Variable", variables, key="q_var")
        q_val = q2.selectbox("Value", domains[q_var], key=f"q_val_{q_var}")
        answer = ASK_LABELS[ask(sym(q_var, q_val))]
        if q_var == "Culprit":
            text = {"Proven true": f"The KB proves {q_val} is the culprit.",
                    "Proven false": f"The KB proves {q_val} is not the culprit.",
                    "Unknown": f"Neither proven guilty nor proven innocent. Guilt not proven does not mean innocence proven."}
        else:
            text = {"Proven true": f"The KB proves {q_var} = {q_val}.",
                    "Proven false": f"The KB proves {q_var} ≠ {q_val}.",
                    "Unknown": f"{q_var} = {q_val} is neither proven nor disproven."}
        show = {"Proven true": st.success, "Proven false": st.error, "Unknown": st.info}[answer]
        show(f"**{answer}.** {text[answer]}")
        st.caption(f"Model checking over {2 ** len(SYMBOLS)} truth assignments; {len(kb_models)} satisfy the KB. "
                   "**Unknown** means some KB models make it true and others make it false.")
        with st.expander("All symbols"):
            st.dataframe(pd.DataFrame([{"Symbol": s, "Result": ASK_LABELS[ask(s)]} for s in SYMBOLS]),
                         hide_index=True, width="stretch")

# ---------------- 6. Hypothesis checker ----------------
with tab_check:
    st.subheader("Check your own theory")
    with st.form("checker"):
        f1, f2 = st.columns(2)
        hc = f1.selectbox("Culprit", SUSPECTS)
        hw = f2.selectbox("Weapon", WEAPONS)
        f3, f4 = st.columns(2)
        hl = f3.selectbox("Location", LOCATIONS)
        ht = f4.selectbox("Time", TIMES)
        if st.form_submit_button("Check hypothesis", width="stretch"):
            h = {"Culprit": hc, "Weapon": hw, "Location": hl, "Time": ht}
            st.session_state["check_result"] = {"h": h, "broken": broken_clues(h)}
    result = st.session_state.get("check_result")
    if result:
        st.markdown(f"**{describe(result['h'])}**")
        if result["broken"]:
            st.error("Not possible. It breaks these encoded constraints:")
            for clue in result["broken"]:
                st.markdown(f"- {clue}")
        else:
            st.success("Consistent with the clues.")
            st.caption("No encoded constraint rules it out. That does not mean it is proven.")

# ---------------- 7. Temporal reasoning ----------------
with tab_hmm:
    st.subheader("When did it happen?")
    hmm, hmm_err = safe_hmm()
    if hmm_err:
        st.error(f"The HMM could not be computed: {hmm_err}")
    else:
        tp = hmm["time_probs"]
        if tp:
            cols = st.columns(len(tp))
            for col, (t, p) in zip(cols, tp.items()):
                col.metric(f"P(murder at {t})", f"{p:.2f}")
        st.dataframe(pd.DataFrame([{
            "Step": STEPS[i],
            "Observation": CLUE_STREAM[i],
            "Viterbi state": hmm["path"][i],
            "P(Alive)": round(hmm["smooth"][i]["Alive"], 3),
            "P(Killed)": round(hmm["smooth"][i]["Killed"], 3),
            "P(Moved)": round(hmm["smooth"][i]["Moved"], 3),
        } for i in range(len(STEPS))]), hide_index=True, width="stretch")
        st.caption("State probabilities are smoothed (forward-backward, using all observations). "
                   "Viterbi is the single most likely sequence; P(murder time) sums over all sequences, "
                   "so the two can point to different times.")

        best_p = max(tp.values()) if tp else None
        hmm_times = [t for t, p in tp.items() if best_p is not None and abs(p - best_p) < 1e-12]
        if res and res["logic"]:
            rows_now, _ = safe_ranking(res["logic"], current_observed())
            if rows_now and hmm_times:
                top_time = rows_now[0]["hypothesis"]["Time"]
                if len(hmm_times) > 1:
                    st.info(f"The HMM is tied between {', '.join(hmm_times)}.")
                elif hmm_times[0] == top_time:
                    st.success(f"HMM most likely time ({hmm_times[0]}) agrees with the top hypothesis time ({top_time}).")
                else:
                    st.warning(f"HMM most likely time ({hmm_times[0]}) disagrees with the top hypothesis time "
                               f"({top_time}). Treat the verdict with care.")
        else:
            st.caption("Solve the case to compare the HMM with the top hypothesis.")
        st.caption("The HMM uses its own fixed observation stream and is kept separate from the Bayesian ranking "
                   "(the 10 PM sighting is already used there), so evidence is not counted twice. "
                   "Changing the evidence controls does not change the HMM.")

# ---------------- 8. Diagnostics ----------------
with tab_diag:
    if res is None:
        st.info("Solve the case to see the diagnostics.")
    else:
        st.dataframe(pd.DataFrame([{
            "Variable": v,
            "Initial": ", ".join(res["initial"][v]),
            "After node consistency": ", ".join(res["unary"][v]),
            "After AC-3": ", ".join(res["ac3"][v]) if res["ac3"] else "(empty)",
        } for v in variables]), hide_index=True, width="stretch")
        st.markdown(
            f"- Combinations: {domain_combinations(res['initial'])} → {domain_combinations(res['unary'])} "
            f"→ {domain_combinations(res['ac3']) if res['ac3'] else 0}\n"
            f"- Backtracking nodes explored: {res['nodes']}\n"
            f"- KB models: {res['kb_count']}"
        )
        ruled_out = [s for s in SUSPECTS if all(h["Culprit"] != s for h in res["logic"])]
        st.markdown("- Suspects ruled out: " + (", ".join(ruled_out) if ruled_out else "none"))
        for v in variables[1:]:
            gone = [x for x in domains[v] if all(h[v] != x for h in res["logic"])]
            st.markdown(f"- {v} values ruled out: " + (", ".join(gone) if gone else "none"))
        with st.expander("Solver log"):
            st.code(res["log"] or "(empty)", language=None)

### 9.2 Generate `requirements.txt`
Only the external libraries `app.py` imports. The tunnel tool is not included because the app does not need it.

In [ ]:
%%writefile requirements.txt
streamlit>=1.52,<2
pandas>=2.0,<4
altair>=5.0,<7

### 9.3 Launch settings
- `LAUNCH_APP = True` installs the requirements and starts the interface (cells 9.4 to 9.6).
- `DOWNLOAD_FILES = True` lets cell 9.9 download `app.py` and `requirements.txt` to your computer.

In [ ]:
LAUNCH_APP = False       # set True to install, start Streamlit and open a tunnel from Colab
DOWNLOAD_FILES = False   # set True to download app.py and requirements.txt in cell 9.9

### 9.4 Install dependencies
Installs `requirements.txt` and downloads `cloudflared` (Cloudflare's tunnel tool) from its official GitHub release.
Runs only when `LAUNCH_APP = True`.

In [ ]:
import os, subprocess, sys, urllib.request

if not LAUNCH_APP:
    print("Skipped (LAUNCH_APP = False).")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    print("Python requirements installed.")
    if not os.path.exists("cloudflared"):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "cloudflared")
        os.chmod("cloudflared", 0o755)
    print("cloudflared ready:", subprocess.run(["./cloudflared", "--version"], capture_output=True, text=True).stdout.strip())

### 9.5 Start Streamlit and the tunnel (optional)
- **Tunnel:** a Cloudflare *quick tunnel* (`trycloudflare.com`). It needs **no account and no token**.
- **It is public, not private.** Anyone who has the URL can open the app while it runs. Don't share the link if you don't want others to see it.
- The URL works only while the Colab runtime, Streamlit and the tunnel are all running. You get a **new URL** after restarting.
- This cell first checks that Streamlit is answering on its health endpoint, then opens the tunnel.
  Running the cell again reuses the server and tunnel it already started instead of creating duplicates.
- It only tracks the processes it starts itself (their PIDs are saved in `ui_pids.json`). Cell 9.8 stops only those.

In [ ]:
import json, os, re, socket, subprocess, sys, time, urllib.request

UI_PID_FILE = "ui_pids.json"
if "UI_PROCS" not in globals():
    UI_PROCS = {"streamlit": None, "tunnel": None, "port": None, "tunnel_port": None, "url": None}


def _ui_alive(proc):
    return proc is not None and proc.poll() is None


def _ui_port_free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def _ui_healthy(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/_stcore/health", timeout=2) as r:
            return r.read().decode().strip() == "ok"
    except Exception:
        return False


def _ui_save_pids():
    pids = {k: UI_PROCS[k].pid for k in ("streamlit", "tunnel") if _ui_alive(UI_PROCS[k])}
    with open(UI_PID_FILE, "w") as f:
        json.dump(pids, f)


if not LAUNCH_APP:
    print("Skipped (LAUNCH_APP = False). Set it to True in cell 9.3 and run cells 9.3 to 9.6.")
else:
    # 1. Streamlit, reused if this notebook already started it
    if _ui_alive(UI_PROCS["streamlit"]):
        print("Streamlit is already running on port", UI_PROCS["port"])
    else:
        _port = 8501
        while not _ui_port_free(_port):   # never kill another program, just use the next free port
            _port += 1
        UI_PROCS["streamlit"] = subprocess.Popen(
            [sys.executable, "-m", "streamlit", "run", "app.py",
             "--server.port", str(_port), "--server.address", "127.0.0.1",
             "--server.headless", "true", "--browser.gatherUsageStats", "false"],
            stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
        UI_PROCS["port"] = _port
        print("Started Streamlit on port", _port)

    _deadline = time.time() + 60
    while not _ui_healthy(UI_PROCS["port"]) and _ui_alive(UI_PROCS["streamlit"]) and time.time() < _deadline:
        time.sleep(1)

    if not _ui_healthy(UI_PROCS["port"]):
        print("Streamlit is not responding. Check streamlit.log. The tunnel was not opened.")
    else:
        print("Streamlit is listening.")
        # 2. tunnel, reused if it still points at the same port
        if _ui_alive(UI_PROCS["tunnel"]) and UI_PROCS["tunnel_port"] == UI_PROCS["port"] and UI_PROCS["url"]:
            print("Tunnel is already running.")
        else:
            if _ui_alive(UI_PROCS["tunnel"]):          # old tunnel to an old port: stop only our own process
                UI_PROCS["tunnel"].terminate()
                UI_PROCS["tunnel"].wait(timeout=10)
            UI_PROCS["url"] = None
            UI_PROCS["tunnel"] = subprocess.Popen(
                ["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{UI_PROCS['port']}"],
                stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
            UI_PROCS["tunnel_port"] = UI_PROCS["port"]
            _deadline = time.time() + 60
            while UI_PROCS["url"] is None and _ui_alive(UI_PROCS["tunnel"]) and time.time() < _deadline:
                time.sleep(1)
                _found = re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
                if _found:
                    UI_PROCS["url"] = _found[0]
            if UI_PROCS["url"] is None:
                print("The tunnel did not give a URL. Check tunnel.log.")
            else:
                print("Tunnel is running.")
        _ui_save_pids()

### 9.6 Interface URL
Open the link (it can take a few seconds to work the first time). It stays the same until the tunnel is stopped or the runtime restarts.

In [ ]:
from IPython.display import HTML, display

if "UI_PROCS" in globals() and UI_PROCS.get("url") and UI_PROCS["tunnel"] and UI_PROCS["tunnel"].poll() is None:
    print("Interface URL:", UI_PROCS["url"])
    display(HTML(f'<a href="{UI_PROCS["url"]}" target="_blank">Open the AI Detective interface</a>'))
    print("Public link: anyone with this URL can open the app while it is running.")
else:
    print("No interface is running. Set LAUNCH_APP = True in 9.3 and run 9.3 to 9.5.")

### 9.7 Check that the generated files are complete
Quick syntax check of `app.py` (it does not start the app).

In [ ]:
import ast

with open("app.py") as f:
    ast.parse(f.read())
print("app.py syntax OK,", sum(1 for _ in open("app.py")), "lines")
print("requirements.txt:", open("requirements.txt").read().split())

### 9.8 Stop the server and tunnel
Stops only the Streamlit and cloudflared processes started by cell 9.5. That includes ones left over from before a
runtime *restart*, which are found through `ui_pids.json` and checked by their command line. No other process is touched.

In [ ]:
import json, os, signal

_stopped = []
if "UI_PROCS" in globals():
    for _k in ("tunnel", "streamlit"):
        _p = UI_PROCS.get(_k)
        if _p is not None and _p.poll() is None:
            _p.terminate()
            try:
                _p.wait(timeout=10)
            except Exception:
                _p.kill()
            _stopped.append(_k)
        UI_PROCS[_k] = None
    UI_PROCS["url"] = None

if os.path.exists("ui_pids.json"):                     # processes from before a kernel restart
    for _k, _pid in json.load(open("ui_pids.json")).items():
        try:
            _cmd = open(f"/proc/{_pid}/cmdline").read().replace("\0", " ")
        except OSError:
            continue                                     # already gone
        if ("streamlit run app.py" in _cmd) or ("cloudflared tunnel" in _cmd):
            os.kill(_pid, signal.SIGTERM)
            _stopped.append(f"{_k} (pid {_pid})")
    os.remove("ui_pids.json")

print("Stopped:", ", ".join(_stopped) if _stopped else "nothing was running")

### 9.9 Download `app.py` and `requirements.txt`
Set `DOWNLOAD_FILES = True` in 9.3 first. These are the two files to put in the root of your GitHub repository.

In [ ]:
if not DOWNLOAD_FILES:
    print("Skipped (DOWNLOAD_FILES = False).")
else:
    try:
        from google.colab import files
        files.download("app.py")
        files.download("requirements.txt")
    except ImportError:
        print("Not running in Colab. The files are in:", os.getcwd())

## 10. Final Verdict

Combines the results of all the parts.

In [14]:
from math import log2

# ---------- 1. Bayesian Network posterior ranking ----------
# Uses exact inference from the explicitly defined Bayesian Network.
# For each observed clue, compare how likely the clue is under this hypothesis
# with how likely it is under the other hypotheses (ratio > 1 supports, < 1 conflicts).
SUPPORT_LIMIT = 1.2      # ratio above this counts as supporting
CONFLICT_LIMIT = 0.8     # ratio below this counts as conflicting

def clue_ratio(h, others, name, value):
    p_h = EVIDENCE[name]["p_true"](h)
    p_o = sum(EVIDENCE[name]["p_true"](o) for o in others) / len(others)
    if not value:
        p_h, p_o = 1 - p_h, 1 - p_o
    return p_h / p_o


def build_ranking(hyps, observed, clue_stream=None):
    """Returns a list of dicts (best first) with everything the interface needs."""
    probs = posterior(hyps, observed)
    time_probs = murder_time_distribution(clue_stream) if clue_stream else {}
    rows = []
    for i, h in enumerate(hyps):
        others = [o for j, o in enumerate(hyps) if j != i]
        supports, conflicts = [], []
        if others:
            for name, value in observed.items():
                ratio = clue_ratio(h, others, name, value)
                if ratio > SUPPORT_LIMIT:
                    supports.append((name, ratio))
                elif ratio < CONFLICT_LIMIT:
                    conflicts.append((name, ratio))
        rows.append({
            "hypothesis": h,
            "probability": probs[i],
            "supports": sorted(supports, key=lambda x: -x[1]),
            "conflicts": sorted(conflicts, key=lambda x: x[1]),
            "hmm_time_prob": time_probs.get(h["Time"]),
        })
    return sorted(rows, key=lambda r: r["probability"], reverse=True)


# ---------- 2. confidence: say when the evidence is not enough ----------
STRONG_TOP, STRONG_MARGIN = 0.80, 0.30
LIKELY_TOP, LIKELY_MARGIN = 0.50, 0.20

def verdict(rows):
    if len(rows) == 0:
        return "NO HYPOTHESIS", "The clues contradict each other."
    if len(rows) == 1:
        return "CONCLUSIVE", "Only one hypothesis is consistent with the clues."
    top, second = rows[0]["probability"], rows[1]["probability"]
    margin = top - second
    if top >= STRONG_TOP and margin >= STRONG_MARGIN:
        return "STRONG", f"Top hypothesis leads by {margin:.2f}."
    if top >= LIKELY_TOP and margin >= LIKELY_MARGIN:
        return "LIKELY", f"Top hypothesis leads by {margin:.2f}, but it is not conclusive."
    return "INCONCLUSIVE", f"Only {margin:.2f} between the top two, more evidence is needed."


# ---------- 3. decision layer: which clue should we look for next? ----------
def entropy(probs):
    return -sum(p * log2(p) for p in probs if p > 0)

def information_gain(hyps, observed, name):
    """Expected drop in entropy if we find out whether clue `name` is True or False."""
    now = posterior(hyps, observed)
    expected = 0.0
    for value in (True, False):
        p_value = sum(now[i] * (EVIDENCE[name]["p_true"](h) if value else 1 - EVIDENCE[name]["p_true"](h))
                      for i, h in enumerate(hyps))
        if p_value > 0:
            after = posterior(hyps, {**observed, name: value})
            expected += p_value * entropy(after)
    return entropy(now) - expected

def next_clue(hyps, observed):
    """The not-yet-known clue with the highest information gain (None if all are known)."""
    options = [(name, information_gain(hyps, observed, name)) for name in EVIDENCE if name not in observed]
    if len(options) == 0:
        return None
    return sorted(options, key=lambda x: -x[1])


# ---------- 4. printing ----------
def show_ranking(rows):
    for rank_no, row in enumerate(rows, start=1):
        print(f"{rank_no}. {describe(row['hypothesis'])}")
        print(f"   P = {row['probability']:.3f}")
        if row["supports"]:
            print("   supported by :", ", ".join(f"{n} (x{r:.1f})" for n, r in row["supports"]))
        if row["conflicts"]:
            print("   conflicts    :", ", ".join(f"{n} (x{r:.1f})" for n, r in row["conflicts"]))
        if row["hmm_time_prob"] is not None:
            print(f"   HMM check    : P(murder at {row['hypothesis']['Time']}) = {row['hmm_time_prob']:.2f}")


# ---------- 5. final verdict ----------
print("Hypotheses at the start :", TOTAL_HYPOTHESES)
print("After CSP               :", len(consistent_hypotheses))
print("After logical inference :", len(logic_hypotheses))
ruled_out = [s for s in SUSPECTS if all(h["Culprit"] != s for h in logic_hypotheses)]
print("Suspects ruled out      :", ruled_out)

final_rows = build_ranking(logic_hypotheses, observed, CLUE_STREAM)
print("\nRanking (all uncertain clues known):")
show_ranking(final_rows)

label, reason = verdict(final_rows)
print(f"\nConfidence: {label}. {reason}")
if len(final_rows) > 0:
    print("Most likely:", describe(final_rows[0]["hypothesis"]), f"(P = {final_rows[0]['probability']:.2f})")

# the HMM is kept separate from the ranking (the 10 PM sighting is already used by the
# Bayesian network), so here we only check that the two agree
if len(final_rows) > 0:
    hmm_time = max(time_probs_all := murder_time_distribution(CLUE_STREAM), key=time_probs_all.get)
    same = final_rows[0]["hypothesis"]["Time"] == hmm_time
    print(f"HMM most likely murder time: {hmm_time}. Top hypothesis time: {final_rows[0]['hypothesis']['Time']}. "
          f"{'They agree.' if same else 'They disagree, so treat the verdict with care.'}")

# the investigation loop: start with no clues and reveal the most useful clue each time
print("\nInvestigation: reveal the most informative clue each step")
known = {}
while True:
    options = next_clue(logic_hypotheses, known)
    rows_now = build_ranking(logic_hypotheses, known)
    label_now, _ = verdict(rows_now)
    top_now = rows_now[0]
    print(f"  known = {list(known) or 'none'}: top = {top_now['hypothesis']['Culprit']} "
          f"(P = {top_now['probability']:.2f}), {label_now}")
    if options is None:
        break
    chosen, gain = options[0]
    print(f"    next clue to look for: {chosen} (information gain {gain:.2f} bits)")
    known[chosen] = observed[chosen]

Hypotheses at the start : 192
After CSP               : 11
After logical inference : 11
Suspects ruled out      : []

Ranking (all uncertain clues known):
1. Rohan (nephew) with the Trophy in the Library at 9 PM
   P = 0.377
   supported by : Fingerprint (x4.7), Motive (x2.0)
   conflicts    : Argument (x0.7), Sighting (x0.8)
   HMM check    : P(murder at 9 PM) = 0.65
2. Meera (secretary) with the Scarf in the Balcony at 10 PM
   P = 0.121
   supported by : Sighting (x4.0)
   conflicts    : Motive (x0.7), Argument (x0.7), Fingerprint (x0.7)
   HMM check    : P(murder at 10 PM) = 0.35
3. Rohan (nephew) with the Knife in the Kitchen at 9 PM
   P = 0.081
   supported by : Motive (x2.0)
   conflicts    : Argument (x0.7), Fingerprint (x0.7), Sighting (x0.8)
   HMM check    : P(murder at 9 PM) = 0.65
4. Rohan (nephew) with the Knife in the Kitchen at 10 PM
   P = 0.081
   supported by : Motive (x2.0)
   conflicts    : Argument (x0.7), Fingerprint (x0.7), Sighting (x0.8)
   HMM check    : P(m

## 11. Testing and Evaluation

- **Constraint consistency:** the CSP should keep only hypotheses that break no clue, and should not remove a valid one.
- **Efficiency:** compare brute force, backtracking only, and node consistency + AC-3 + backtracking.
  For brute force, explored = full hypotheses checked. For backtracking, explored = values tried.

In [15]:
# checks every full combination one by one
def brute_force(doms):
    solutions = []
    checked = 0
    for values in product(*[doms[var] for var in variables]):
        h = dict(zip(variables, values))
        checked += 1
        if is_consistent(h):
            solutions.append(h)
    return solutions, checked


def check(name, passed):
    print(("PASS  " if passed else "FAIL  ") + name)


# actual answer of our case, only used for testing
TRUE_ANSWER = {"Culprit": "Rohan", "Weapon": "Trophy", "Location": "Library", "Time": "9 PM"}

all_hypotheses = [dict(zip(variables, values)) for values in product(*[domains[var] for var in variables])]
eliminated = [h for h in all_hypotheses if h not in consistent_hypotheses]
print(f"total {len(all_hypotheses)}, kept {len(consistent_hypotheses)}, eliminated {len(eliminated)}\n")

check("every kept hypothesis breaks no clue",
      all(is_consistent(h) for h in consistent_hypotheses))
check("every eliminated hypothesis breaks at least one clue",
      all(len(broken_clues(h)) > 0 for h in eliminated))

bf_solutions, bf_checked = brute_force(domains)
check("CSP gives the same hypotheses as brute force",
      len(bf_solutions) == len(consistent_hypotheses) and all(h in bf_solutions for h in consistent_hypotheses))
check("true answer is not eliminated",
      TRUE_ANSWER in consistent_hypotheses)

total 192, kept 11, eliminated 181

PASS  every kept hypothesis breaks no clue
PASS  every eliminated hypothesis breaks at least one clue
PASS  CSP gives the same hypotheses as brute force
PASS  true answer is not eliminated


In [16]:
start = time.perf_counter()
sol1, explored1 = brute_force(domains)
time1 = (time.perf_counter() - start) * 1000

start = time.perf_counter()
sol2, explored2 = backtracking_search(domains)
time2 = (time.perf_counter() - start) * 1000

start = time.perf_counter()
reduced = ac3(apply_unary_constraints(domains))
sol3, explored3 = backtracking_search(reduced)
time3 = (time.perf_counter() - start) * 1000

print(f"{'Method':<32}{'Explored':>10}{'Solutions':>11}{'Time (ms)':>11}")
print(f"{'Brute force':<32}{explored1:>10}{len(sol1):>11}{time1:>11.3f}")
print(f"{'Backtracking only':<32}{explored2:>10}{len(sol2):>11}{time2:>11.3f}")
print(f"{'Unary + AC-3 + backtracking':<32}{explored3:>10}{len(sol3):>11}{time3:>11.3f}")

Method                            Explored  Solutions  Time (ms)
Brute force                            192         11      2.681
Backtracking only                       92         11      0.808
Unary + AC-3 + backtracking             67         11      0.989


**Logical inference tests:** the KB should be consistent, give the same hypotheses as the CSP, prove the facts AC-3 removed, and never eliminate the true answer.

In [17]:
# Tests for logical inference
check("KB is satisfiable (the clues do not contradict)", len(kb_models) > 0)
check("every KB model has exactly one value per variable",
      all(exactly_one(var)(m) for m in kb_models for var in variables))
check("number of KB models equals number of CSP solutions", len(kb_models) == len(consistent_hypotheses))
check("logic keeps the same hypotheses as the CSP",
      len(logic_hypotheses) == len(consistent_hypotheses) and all(h in consistent_hypotheses for h in logic_hypotheses))
check("KB proves Kabir is innocent", entails(lambda m: not m["C_Kabir"]))
check("KB does not prove any single culprit", all(ask(sym("Culprit", s)) != "TRUE (proven)" for s in SUSPECTS))
check("KB rejects a hypothesis that breaks a clue",
      entails(lambda m: not (m["C_Kabir"] and m["W_Knife"] and m["L_Kitchen"] and m["T_9PM"])))
check("true answer is not eliminated by logic", TRUE_ANSWER in logic_hypotheses)

# entailment test on a small KB:  {P, P => Q} entails Q but not NOT Q
small = [m for m in [{"P": p, "Q": q} for p in [True, False] for q in [True, False]]
         if m["P"] and implies(m["P"], m["Q"])]
check("modus ponens: {P, P => Q} entails Q", all(m["Q"] for m in small))

PASS  KB is satisfiable (the clues do not contradict)
PASS  every KB model has exactly one value per variable
PASS  number of KB models equals number of CSP solutions
PASS  logic keeps the same hypotheses as the CSP
FAIL  KB proves Kabir is innocent
PASS  KB does not prove any single culprit
FAIL  KB rejects a hypothesis that breaks a clue
PASS  true answer is not eliminated by logic
PASS  modus ponens: {P, P => Q} entails Q


**Bayesian network tests:** posterior probabilities should sum to 1, no evidence should produce equal prior probabilities, the true answer should rank first, clue order should not affect results, evidence should favour the appropriate suspects, and sensitivity and ablation tests should demonstrate how changing or removing clues affects the ranking.

In [18]:
import random

# Tests for the Bayesian network
check("posteriors sum to 1", abs(sum(p for h, p in ranked_hypotheses) - 1) < 1e-9)
check("no evidence gives the prior (equal probabilities)",
      all(abs(p - 1 / len(logic_hypotheses)) < 1e-9 for p in posterior(logic_hypotheses, {})))
check("true answer is ranked first", ranked_hypotheses[0][0] == TRUE_ANSWER)

names = list(EVIDENCE)
random.shuffle(names)
check("order of clues does not change the result",
      all(abs(a - b) < 1e-12 for a, b in
          zip(posterior(logic_hypotheses, observed),
              posterior(logic_hypotheses, {n: True for n in names}))))

rohan = [h for h in logic_hypotheses if h["Culprit"] == "Rohan"][0]
anita = [h for h in logic_hypotheses if h["Culprit"] == "Anita"][0]
check("fingerprint favours Rohan over Anita", bayes_factor(rohan, anita, "Fingerprint") > 1)
check("argument favours Anita over Rohan", bayes_factor(anita, rohan, "Argument") > 1)

# sensitivity: change the fingerprint reliability and see whether the answer changes
saved = EVIDENCE["Fingerprint"]["p_true"]
print("\nSensitivity to the fingerprint reliability:")
for r in [0.3, 0.5, 0.7, 0.9]:
    EVIDENCE["Fingerprint"]["p_true"] = lambda h, r=r: (
        r if (h["Culprit"] == "Rohan" and h["Weapon"] == "Trophy") else 0.15)
    best = rank_hypotheses(logic_hypotheses, observed)[0]
    print(f"  reliability {r}: top = {best[0]['Culprit']} (P = {best[1]:.2f})")
EVIDENCE["Fingerprint"]["p_true"] = saved

# ablation: drop one clue at a time
print("\nTop hypothesis when one clue is removed:")
for name in EVIDENCE:
    rest = {n: True for n in EVIDENCE if n != name}
    best = rank_hypotheses(logic_hypotheses, rest)[0]
    print(f"  without {name:<12}: {best[0]['Culprit']} (P = {best[1]:.2f})")

PASS  posteriors sum to 1
PASS  no evidence gives the prior (equal probabilities)
PASS  true answer is ranked first
PASS  order of clues does not change the result
FAIL  fingerprint favours Rohan over Anita
PASS  argument favours Anita over Rohan

Sensitivity to the fingerprint reliability:
  reliability 0.3: top = Rohan (P = 0.21)
  reliability 0.5: top = Rohan (P = 0.30)
  reliability 0.7: top = Rohan (P = 0.38)
  reliability 0.9: top = Rohan (P = 0.44)

Top hypothesis when one clue is removed:
  without Motive      : Rohan (P = 0.21)
  without Fingerprint : Meera (P = 0.17)
  without Argument    : Rohan (P = 0.43)
  without Sighting    : Rohan (P = 0.42)


**HMM tests:** Initial, transition, and emission probabilities should sum to 1, filtering beliefs should remain normalised, known clues should correctly identify Vikram as alive at 8 PM, Viterbi should match brute-force sequence search, impossible state transitions should never occur, and murder-time probabilities should sum to 1. Sensitivity and ablation tests evaluate how changes in observations and transition probabilities affect the predicted murder time.

In [19]:
from itertools import product

# Tests for the HMM
def path_probability(seq, obs):
    p = START[seq[0]] * EMIT[seq[0]][obs[0]]
    for i in range(1, len(seq)):
        p = p * TRANS[seq[i - 1]][seq[i]] * EMIT[seq[i]][obs[i]]
    return p

obs = CLUE_STREAM
check("start, transition and emission rows each sum to 1",
      abs(sum(START.values()) - 1) < 1e-9 and
      all(abs(sum(row.values()) - 1) < 1e-9 for row in list(TRANS.values()) + list(EMIT.values())))
check("every belief from filtering sums to 1",
      all(abs(sum(b.values()) - 1) < 1e-9 for b in forward(obs)))
check("Vikram is alive at 8 PM (clue 3)", forward_backward(obs)[0]["Alive"] > 0.999)
check("last filtering belief equals last smoothing belief",
      all(abs(forward(obs)[-1][s] - forward_backward(obs)[-1][s]) < 1e-9 for s in STATES))

brute_best = max(product(STATES, repeat=len(obs)), key=lambda seq: path_probability(seq, obs))
check("Viterbi equals brute force over all 3^4 sequences", list(brute_best) == viterbi(obs)[0])
check("Viterbi path never goes back to Alive after Killed",
      all(not (viterbi(obs)[0][i] != "Alive" and viterbi(obs)[0][i + 1] == "Alive") for i in range(len(obs) - 1)))
check("murder time probabilities sum to 1", abs(sum(murder_time_distribution(obs).values()) - 1) < 1e-9)

# ablation: what if the 10 PM clue were just "Normal" instead of "Suspicious"?
quiet = ["SeenAlive", "Normal", "Normal", "BodyFound"]
print("\nWith the 10 PM clue = Normal:")
print("  Viterbi:", viterbi(quiet)[0])
print("  P(murder time):", {k: round(v, 3) for k, v in murder_time_distribution(quiet).items()})

# sensitivity: change the chance of being killed in each hour
print("\nSensitivity to P(Alive -> Killed):")
saved = dict(TRANS["Alive"])
for h in [0.3, 0.5, 0.7]:
    TRANS["Alive"] = {"Alive": 1 - h, "Killed": h, "Moved": 0.0}
    d = murder_time_distribution(obs)
    print(f"  {h}: P(9 PM) = {d['9 PM']:.2f}, P(10 PM) = {d['10 PM']:.2f}, Viterbi murder at {STEPS[viterbi(obs)[0].index('Killed')]}")
TRANS["Alive"] = saved

PASS  start, transition and emission rows each sum to 1
PASS  every belief from filtering sums to 1
PASS  Vikram is alive at 8 PM (clue 3)
PASS  last filtering belief equals last smoothing belief
PASS  Viterbi equals brute force over all 3^4 sequences
PASS  Viterbi path never goes back to Alive after Killed
PASS  murder time probabilities sum to 1

With the 10 PM clue = Normal:
  Viterbi: ['Alive', 'Killed', 'Moved', 'Moved']
  P(murder time): {'9 PM': 0.782, '10 PM': 0.218}

Sensitivity to P(Alive -> Killed):
  0.3: P(9 PM) = 0.57, P(10 PM) = 0.43, Viterbi murder at 10 PM
  0.5: P(9 PM) = 0.65, P(10 PM) = 0.35, Viterbi murder at 10 PM
  0.7: P(9 PM) = 0.75, P(10 PM) = 0.25, Viterbi murder at 9 PM


**Hypothesis ranking tests:** Rankings should be sorted by descending probability, retain all hypotheses from the logical inference stage, and have probabilities summing to 1. The true answer should rank first and be supported by relevant evidence. Verdict classification should correctly handle inconclusive, likely, and conclusive cases. Information gain should never be negative, and the next-clue function should stop when all clues are known. Ablation testing compares rule-only reasoning (CSP + logic) with the combined probabilistic system, while incremental clue testing evaluates changes in confidence as evidence is added.

In [20]:
# Tests for the ranking
check("ranking is sorted from highest to lowest probability",
      all(final_rows[i]["probability"] >= final_rows[i + 1]["probability"] for i in range(len(final_rows) - 1)))
check("ranking has the same hypotheses as the logic stage",
      len(final_rows) == len(logic_hypotheses) and all(r["hypothesis"] in logic_hypotheses for r in final_rows))
check("ranking probabilities sum to 1", abs(sum(r["probability"] for r in final_rows) - 1) < 1e-9)
check("true answer is ranked first", final_rows[0]["hypothesis"] == TRUE_ANSWER)
check("the winner is supported by the fingerprint clue",
      "Fingerprint" in [n for n, r in final_rows[0]["supports"]])
check("no clues gives an INCONCLUSIVE verdict", verdict(build_ranking(logic_hypotheses, {}))[0] == "INCONCLUSIVE")
check("all four clues give a LIKELY verdict", verdict(final_rows)[0] == "LIKELY")
check("verdict handles zero and one hypothesis",
      verdict([])[0] == "NO HYPOTHESIS" and verdict(build_ranking(logic_hypotheses[:1], {}))[0] == "CONCLUSIVE")
check("information gain is never negative",
      all(information_gain(logic_hypotheses, {}, n) >= -1e-12 for n in EVIDENCE))
check("next_clue returns None when every clue is known", next_clue(logic_hypotheses, observed) is None)

# ablation from the proposal: a simple baseline (logic only, no probabilities) vs the full system
print("\nAblation: rule-only baseline vs combined system")
print(f"  rule-only (CSP + logic): {len(logic_hypotheses)} hypotheses left, no way to pick one")
print(f"  combined               : top = {final_rows[0]['hypothesis']['Culprit']} "
      f"(P = {final_rows[0]['probability']:.2f}), {verdict(final_rows)[0]}")

# how the candidate set and confidence change as clues are added (evaluation item in the proposal)
print("\nProbability of the true answer as clues are added:")
order = list(EVIDENCE)
for k in range(len(order) + 1):
    partial = {n: True for n in order[:k]}
    true_p = [r["probability"] for r in build_ranking(logic_hypotheses, partial) if r["hypothesis"] == TRUE_ANSWER][0]
    print(f"  {k} clues {order[:k] if k else '[]'}: P(true answer) = {true_p:.3f}")


# Bayesian Network validation

check(
    "Bayesian Network contains all four evidence nodes",
    set(EVIDENCE_NODES) == {
        "Motive", "Fingerprint", "Argument", "Sighting"
    }
)

check(
    "all CPT probabilities lie between 0 and 1",
    all(
        0 <= EVIDENCE[name]["p_true"](h) <= 1
        for h in logic_hypotheses
        for name in EVIDENCE_NODES
    )
)

check(
    "each evidence CPT outcome pair sums to 1",
    all(
        abs(
            cpt_probability(h, name, True)
            + cpt_probability(h, name, False) - 1
        ) < 1e-9
        for h in logic_hypotheses
        for name in EVIDENCE_NODES
    )
)

check(
    "no evidence gives equal prior probabilities",
    all(
        abs(p - 1 / len(logic_hypotheses)) < 1e-9
        for p in posterior(logic_hypotheses, {})
    )
)

check(
    "posterior probabilities sum to 1",
    abs(sum(posterior(logic_hypotheses, observed)) - 1) < 1e-9
)

# Compare the new implementation with the original
# direct likelihood formulation as a numerical consistency check.

old_style_scores = []

for h in logic_hypotheses:
    score = 1.0 / len(logic_hypotheses)

    for name, value in observed.items():
        p = EVIDENCE[name]["p_true"](h)
        score *= p if value else 1 - p

    old_style_scores.append(score)

z = sum(old_style_scores)
old_style_posterior = [p / z for p in old_style_scores]

new_style_posterior = posterior(logic_hypotheses, observed)

check(
    "Bayesian Network matches factorised exact inference",
    all(
        abs(a - b) < 1e-10
        for a, b in zip(old_style_posterior, new_style_posterior)
    )
)

check(
    "clue order does not affect posterior",
    all(
        abs(a - b) < 1e-10
        for a, b in zip(
            posterior(logic_hypotheses, observed),
            posterior(
                logic_hypotheses,
                dict(reversed(list(observed.items())))
            )
        )
    )
)


PASS  ranking is sorted from highest to lowest probability
PASS  ranking has the same hypotheses as the logic stage
PASS  ranking probabilities sum to 1
PASS  true answer is ranked first
PASS  the winner is supported by the fingerprint clue
PASS  no clues gives an INCONCLUSIVE verdict
FAIL  all four clues give a LIKELY verdict
PASS  verdict handles zero and one hypothesis
PASS  information gain is never negative
PASS  next_clue returns None when every clue is known

Ablation: rule-only baseline vs combined system
  rule-only (CSP + logic): 11 hypotheses left, no way to pick one
  combined               : top = Rohan (P = 0.38), INCONCLUSIVE

Probability of the true answer as clues are added:
  0 clues []: P(true answer) = 0.091
  1 clues ['Motive']: P(true answer) = 0.167
  2 clues ['Motive', 'Fingerprint']: P(true answer) = 0.483
  3 clues ['Motive', 'Fingerprint', 'Argument']: P(true answer) = 0.415
  4 clues ['Motive', 'Fingerprint', 'Argument', 'Sighting']: P(true answer) = 0.377
P

## 12. Conclusion

(Write after all the parts are added.)